# Rerun 2 · Plain-backbone baseline and repeated full model under the ablation protocol (review points 34, 80, and 12/13 support)

**What this fixes.**
* **Point 34:** the architecture-ablation table has no plain backbone (spatial branch only, no morphology branch, no fusion, no attention). This notebook trains it with the identical recipe.
* **Points 80 / 12 / 13:** the full model is trained **again from a fresh initialisation, for each seed in `SEED_LIST`, alongside the ablation arms**, so the ablation deltas can be compared with a baseline from the same procedure and with the run-to-run spread of the same seed.
* Optional (`RUN_REMOVAL_ARMS=True`): re-runs the four removal arms of the architecture table (− spatial, − morphology, − gated fusion, − MS-CA-ECA) under the same loop.

Everything uses the SMGF_Net_7 recipe (flip-TTA or single pass chosen on validation, Mixup + CutMix, 60 epochs, patience 8, seed set per run). Outputs: `plain_backbone_results.csv`, `plain_backbone_predictions.npz`, `ablation_arm_summary.csv` and McNemar tests of each arm against the freshly trained full model.

**Runs.** 3 seeds (42, 13, 77) x 15 arms = 45 runs, in three groups. *Model breakdown:* full model, plain backbone, and removal of the spatial branch, morphology branch, gated fusion or MS-CA-ECA. *MS-CA-ECA breakdown:* removal of the multi-scale context, the coordinate attention or the ECA (optionally each part alone, `RUN_MSCAECA_ONLY_ARMS`). *Loss breakdown:* plain cross-entropy, and CAWL+ without class weights, label smoothing, the confidence penalty or the prototype margin (`RUN_LOSS_ARMS`). About 6 GPU hours per seed at 0.4 h per run (not measured), so run one seed per session. The results file is appended after every run and finished runs are skipped.

**Attach:** BRISC2025 only.


**Revision (review points on seeds and leakage).** This notebook uses the SMGF_Net_7 setup: 288 px letterbox preprocessing, lesion-guided SMGF-Net, Mixup + CutMix, source-balanced sampling, near-duplicate cleaning and a grouped validation split, 60 epochs with patience 8 and checkpoint selection on validation loss. `SEED` is set per run, so a seed re-draws initialisation, augmentation and the grouped validation split; the test set is fixed. `run_experiment` also writes `<run>_predictions.npz` and chooses the headline inference mode on validation only. Notebook 09 aggregates the CSV files.

In [ ]:
import time

# ---- time budget: Kaggle ends a session at 12 h and a session that is cut off may lose its output files ----
TIME_BUDGET_H = 11.0      # no new run is started if it could not finish inside this many hours since the first cell started
_T0 = time.time(); _RUN_H = []; _STOP_PRINTED = False
def can_start_run():
    global _STOP_PRINTED
    used = (time.time() - _T0) / 3600
    need = 1.15 * (max(_RUN_H) if _RUN_H else 0.6)      # longest finished run so far (0.6 h before the first one)
    ok = used + need <= TIME_BUDGET_H
    if not ok and not _STOP_PRINTED:
        _STOP_PRINTED = True
        print(f"[time budget] {used:.1f} h used, the next run needs about {need:.1f} h: no further run is started. "
              "Results so far are saved. Save this version, attach its output as a dataset, set RESUME_DIR and run again.")
    return ok
SEED_LIST        = [42, 13, 77]   # three seeds; 15 arms x 3 seeds = 45 runs with the defaults below (about 6 GPU h per seed, so run one seed per session)
RUN_REMOVAL_ARMS = True           # also runs -Spatial, -Morphology, -GatedFusion, -MS-CA-ECA (5 removal arms incl. the lesion head, x 3 seeds = 15 runs)
RUN_MSCAECA_PARTS = True          # MS-CA-ECA breakdown: -Multiscale, -Coord, -ECA (3 arms x 3 seeds = 9 runs)
RUN_MSCAECA_ONLY_ARMS = False    # also each part alone: Multiscale-only, Coord-only, ECA-only (+9 runs)
RUN_LOSS_ARMS     = True          # CAWL+ breakdown: plain CE, -class weights, -label smoothing, -confidence penalty, -prototype margin (5 arms x 3 seeds = 15 runs)
# Results of an earlier session used the old recipe, so do not reuse them. A stopped session resumes from the files saved after every run:
# attach that session's output as a Kaggle dataset and set RESUME_DIR = "/kaggle/input/<your-dataset-name>".
RESUME_DIR = ""   # every attached input is searched for plain_backbone_results.csv + plain_backbone_predictions.npz (the pair with the most finished runs is used); "" = start from scratch


## §1 · Definitions reused unchanged from the stage-1 notebook (same split, preprocessing, augmentation, loss, Mixup, EMA, flip-TTA, schedule, seed)

Preprocessing is the stage-1 `preprocess()` (percentile clip + min-max stretch, **no AMEF**), the pipeline that produced the reported centralized result.

## Foundation: the SMGF_Net_7 setup
Configuration, preprocessing (aspect-preserving resize to 288 px, robust normalisation), augmentation with lesion-safe geometry, dataset, SMGF-Net + MS-CA-ECA with the training-only lesion head, CAWL+ loss, near-duplicate cleaning and grouped validation split, training utilities and the experiment runner. The cells are identical in every notebook of this repository.

In [ ]:
!pip install -q -U timm scikit-learn thop scikit-image

In [ ]:
import os, math, random, warnings, hashlib
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # needed for deterministic cuBLAS; set before the first CUDA call
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                              precision_score, recall_score, roc_auc_score,
                              confusion_matrix, classification_report,
                              matthews_corrcoef, cohen_kappa_score)
import matplotlib.pyplot as plt
import timm
import pandas as pd
from scipy.stats import spearmanr
warnings.filterwarnings("ignore")

SEED = 42
RUN_MULTISEED = False   # True = also train seeds 13/42/77 again (3 more full runs), then the seed summary and ensemble cells
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
# Determinism flags (reviewer item: same seed 42 gave 96.28 / 93.47 / 93.87). These remove cuDNN autotuning and
# non-deterministic kernels where PyTorch has a deterministic alternative; AMP and DataLoader worker order can still
# introduce run-to-run differences, so report repeat-run spread rather than assuming bitwise reproducibility.
DETERMINISTIC = True
if DETERMINISTIC:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE    = 288   # was 224. Remaining errors are glioma<->meningioma; BRISC images are ~512 px, so 224 throws away margin/texture detail. Parameter count is unchanged (all-conv + GAP).
NUM_CLASSES = 4
CLASS_NAMES  = ["glioma", "meningioma", "notumor", "pituitary"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".jfif", ".tif", ".tiff", ".webp"}

OUT_DIR  = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE  = 32
NUM_WORKERS = 2

DROPOUT_RATE  = 0.3
USE_AMP       = True
EARLY_STOP_PATIENCE = 15   # epochs with no val AUC improvement before stopping

print("Device:", DEVICE)
print("Classes:", CLASS_NAMES)
print(f"DROPOUT_RATE={DROPOUT_RATE}  USE_AMP={USE_AMP}")

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def read_gray(path: str) -> np.ndarray:
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise ValueError(f"Cannot read: {path}")
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY if img.shape[-1] == 3
                           else cv2.COLOR_BGRA2GRAY)
    return img

def robust_normalize(img: np.ndarray) -> np.ndarray:
    img = img.astype(np.float32)
    p1, p99 = np.percentile(img, [1, 99])
    img = np.clip(img, p1, p99)
    img -= img.min(); img /= (img.max() + 1e-8)
    return img

def to_3ch_tensor(img01: np.ndarray) -> torch.Tensor:
    rgb = np.stack([img01, img01, img01], axis=2)
    rgb = torch.tensor(rgb).permute(2, 0, 1).float()
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (rgb - mean) / std


# Decoded + resized uint8 images, keyed by (path, IMG_SIZE). Filled in the main process by
# warm_gray_cache() (called from BrainTumorDataset.__init__) BEFORE the DataLoader forks its workers, so
# workers inherit it. Output of preprocess() is bit-identical to the uncached version; this only removes the
# per-epoch JPEG decode + resize, which is what lets a larger IMG_SIZE and more epochs fit in the Kaggle quota.
_GRAY_CACHE: Dict[Tuple[str, int, bool], np.ndarray] = {}

PAD_TO_SQUARE = True     # aspect-preserving resize (long side -> IMG_SIZE, background padding). False = the old square stretch
_NATIVE_SHAPE: Dict[str, Tuple[int, int]] = {}      # path -> (height, width) of the file on disk, filled when an image is decoded

def letterbox(a: np.ndarray, size: int, interp: int, pad: int = 0) -> np.ndarray:
    """Resize a 2-D array to size x size. Square inputs give the same result as cv2.resize; non-square inputs keep their aspect
    ratio and are centred on a `pad`-valued canvas (images: background 0, masks: 0), so anatomy is not stretched."""
    h, w = a.shape[:2]
    if h == w or not PAD_TO_SQUARE:
        return cv2.resize(a, (size, size), interpolation=interp)
    s = size / float(max(h, w))
    nh, nw = max(1, int(round(h * s))), max(1, int(round(w * s)))
    r = cv2.resize(a, (nw, nh), interpolation=interp)
    out = np.full((size, size), pad, dtype=r.dtype)
    y0, x0 = (size - nh) // 2, (size - nw) // 2
    out[y0:y0 + nh, x0:x0 + nw] = r
    return out

def _load_resized_gray(path: str) -> np.ndarray:
    g = _GRAY_CACHE.get((path, IMG_SIZE, PAD_TO_SQUARE))
    if g is None:
        _raw = read_gray(path); _NATIVE_SHAPE[path] = _raw.shape[:2]
        g = letterbox(_raw, IMG_SIZE, cv2.INTER_AREA)
    return g

def warm_gray_cache(samples) -> None:
    for s in samples:
        k = (s["path"], IMG_SIZE, PAD_TO_SQUARE)
        if k not in _GRAY_CACHE:
            _GRAY_CACHE[k] = _load_resized_gray(s["path"])


def preprocess(path: str) -> torch.Tensor:
    """Returns a single [3,IMG_SIZE,IMG_SIZE] image tensor.
    Pipeline: read_gray -> resize -> robust_normalize -> tensor."""
    gray = _load_resized_gray(path)
    img01 = robust_normalize(gray)
    return to_3ch_tensor(img01)

# ---- Lesion masks (BRISC2025 segmentation_task). Used ONLY as a training signal for the auxiliary lesion head;
# never read for validation/test samples and never an input to the classifier. MASK_INDEX is filled in the §11 path cell.
MASK_INDEX: Dict[Tuple[str, str], str] = {}
_MASK_CACHE: Dict[Tuple[str, int, bool], np.ndarray] = {}

def _norm_stem(stem: str) -> str:
    s = stem.lower()
    for suf in ("_mask", "-mask", "_seg", "_label", "_gt"):
        if s.endswith(suf): s = s[: -len(suf)]
    return s

def _split_of(path: str) -> str:
    parts = Path(path).parts
    return "train" if "train" in parts else ("test" if "test" in parts else "other")

def mask_path_of(s: Dict) -> Optional[str]:
    return MASK_INDEX.get((_split_of(s["path"]), _norm_stem(Path(s["path"]).stem)))

def _load_mask_arr(mp: str) -> np.ndarray:
    k = (mp, IMG_SIZE, PAD_TO_SQUARE)
    m = _MASK_CACHE.get(k)
    if m is None:
        raw = read_gray(mp)
        m = (letterbox(raw, IMG_SIZE, cv2.INTER_NEAREST) > 0).astype(np.uint8)
        _MASK_CACHE[k] = m
    return m

def warm_mask_cache(samples) -> None:
    for s in samples:
        mp = mask_path_of(s)
        if mp is not None: _load_mask_arr(mp)

def get_mask(s: Dict) -> Tuple[torch.Tensor, float]:
    """(mask [1,H,W] float in {0,1}, valid). no-tumour images get an empty mask (valid); a tumour image without a mask
    file is marked invalid and skipped by the lesion loss."""
    if s["class_name"] == "notumor":
        return torch.zeros(1, IMG_SIZE, IMG_SIZE), 1.0
    mp = mask_path_of(s)
    if mp is None:
        return torch.zeros(1, IMG_SIZE, IMG_SIZE), 0.0
    return torch.from_numpy(_load_mask_arr(mp).astype(np.float32)).unsqueeze(0), 1.0

print("Preprocessing ready.")

In [ ]:
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode

class MRIAugment:
    """Training-only augmentation on the normalised [3,H,W] tensor.
    v3: adds scale + shear to the affine step, Gaussian pixel noise, and fills rotated/translated borders with the
    image's own background value instead of 0 (0 is mid-grey after ImageNet normalisation, so earlier versions
    pasted grey wedges that never occur in test images). The earlier robustness run showed accuracy collapsing
    under mild noise/blur, i.e. the net leans on fine pixel texture; noise and blur augmentation target that."""
    def __init__(self, p_flip=0.5, max_rotate=10.0, p_rotate=0.5,
                 brightness=0.2, contrast=0.2, p_jitter=0.5,
                 translate=0.06, p_translate=0.5, scale_range=(0.90, 1.10), shear=4.0,
                 use_gamma=True, gamma_range=(0.85, 1.15), p_gamma=0.3,
                 p_noise=0.25, noise_std=(0.02, 0.08), p_blur=0.15, p_erase=0.0):
        self.p_flip = p_flip
        self.max_rotate = max_rotate; self.p_rotate = p_rotate
        self.brightness = brightness; self.contrast = contrast; self.p_jitter = p_jitter
        self.translate = translate; self.p_translate = p_translate
        self.scale_range = scale_range; self.shear = shear
        self.use_gamma = use_gamma; self.gamma_range = gamma_range; self.p_gamma = p_gamma
        self.p_noise = p_noise; self.noise_std = noise_std; self.p_blur = p_blur
        self.p_erase = p_erase     # random erasing: one 2-8 % patch is replaced by the background value (set per profile in the A/B cell)

    @staticmethod
    def _keeps_lesion(before: torch.Tensor, after: torch.Tensor, min_keep: float = 0.9) -> bool:
        """Lesion-safe geometry: a rotation / affine step is accepted only if at least 90 % of the lesion area stays in frame.
        Images without a lesion (empty mask) are never restricted."""
        a0 = float(before.sum())
        return a0 <= 0.0 or float(after.sum()) >= min_keep * a0

    def __call__(self, img: torch.Tensor, mask: Optional[torch.Tensor] = None):
        """img [3,H,W]; optional mask [1,H,W] receives the SAME flip / rotation / affine (nearest neighbour, fill 0).
        Returns img, or (img, mask) when a mask is given. Intensity terms and erasing touch the image only."""
        fill = img.amin(dim=(1, 2)).tolist()            # per-channel background value
        if random.random() < self.p_flip:
            img = TF.hflip(img)
            if mask is not None: mask = TF.hflip(mask)
        if random.random() < self.p_rotate:
            angle = random.uniform(-self.max_rotate, self.max_rotate)
            img_before = img
            img = TF.rotate(img, angle, fill=fill)
            if mask is not None:
                _m2 = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST, fill=[0.0])
                if not self._keeps_lesion(mask, _m2): img = img_before; _m2 = mask      # rotation would push the lesion out of frame: skip it
                mask = _m2
        if random.random() < self.p_translate:
            tx = random.uniform(-self.translate, self.translate) * IMG_SIZE
            ty = random.uniform(-self.translate, self.translate) * IMG_SIZE
            sc = random.uniform(*self.scale_range)
            sh = random.uniform(-self.shear, self.shear)
            img_before = img
            img = TF.affine(img, angle=0, translate=[int(tx), int(ty)], scale=sc, shear=[sh, 0.0], fill=fill)
            if mask is not None:
                _m2 = TF.affine(mask, angle=0, translate=[int(tx), int(ty)], scale=sc, shear=[sh, 0.0],
                                interpolation=InterpolationMode.NEAREST, fill=[0.0])
                if not self._keeps_lesion(mask, _m2): img = img_before; _m2 = mask      # translation / scale would cut the lesion: skip it
                mask = _m2
        if random.random() < self.p_jitter:
            b = 1.0 + random.uniform(-self.brightness, self.brightness)
            c = 1.0 + random.uniform(-self.contrast, self.contrast)
            img = torch.clamp(img * c + (b - 1.0), -3.0, 3.0)
        if self.use_gamma and random.random() < self.p_gamma:
            gamma = random.uniform(*self.gamma_range)
            img = torch.sign(img) * (torch.abs(img) ** gamma)
        if random.random() < self.p_blur:
            img = TF.gaussian_blur(img, kernel_size=3, sigma=random.uniform(0.3, 0.8))
        if random.random() < self.p_noise:
            img = img + torch.randn_like(img) * random.uniform(*self.noise_std)
        if self.p_erase > 0 and random.random() < self.p_erase:
            img = img.clone()                                   # never write into a cached tensor
            _, h, w = img.shape
            area = h * w * random.uniform(0.02, 0.08); ar = random.uniform(0.5, 2.0)
            eh = int(min(h - 1, round((area * ar) ** 0.5))); ew = int(min(w - 1, round((area / ar) ** 0.5)))
            y0 = random.randint(0, h - eh); x0 = random.randint(0, w - ew)
            img[:, y0:y0 + eh, x0:x0 + ew] = torch.tensor(fill, dtype=img.dtype, device=img.device).view(-1, 1, 1)
        return img if mask is None else (img, mask)

train_augment = MRIAugment()
print("MRIAugment ready (v3: scale/shear, background fill, noise, light blur).")

In [ ]:
class BrainTumorDataset(Dataset):
    def __init__(self, samples: List[Dict], is_train: bool = False,
                 augmenter: Optional[MRIAugment] = None):
        self.samples   = samples
        warm_gray_cache(samples)   # main process, before any DataLoader workers exist
        if is_train: warm_mask_cache(samples)
        self.is_train  = is_train
        self.augmenter = augmenter

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = preprocess(s["path"])
        mask, valid = get_mask(s) if self.is_train else (torch.zeros(1, IMG_SIZE, IMG_SIZE), 0.0)
        if self.is_train and self.augmenter is not None:
            img, mask = self.augmenter(img, mask)
        return {"image": img, "label": torch.tensor(s["label"], dtype=torch.long),
                "class_name": s["class_name"], "path": s["path"],
                "mask": mask, "mask_valid": torch.tensor(valid, dtype=torch.float32)}

print("BrainTumorDataset ready.")

In [ ]:
class ECA(nn.Module):
    """Efficient Channel Attention (Wang et al., 2020). ~5-9 parameters."""
    def __init__(self, channels: int, gamma: int = 2, b: int = 1):
        super().__init__()
        t = int(abs((math.log2(channels) + b) / gamma))
        k = t if t % 2 else t + 1
        k = max(k, 3)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k, padding=(k - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.avg_pool(x)
        y = y.squeeze(-1).transpose(-1, -2)
        y = self.conv(y)
        y = self.sigmoid(y).transpose(-1, -2).unsqueeze(-1)
        return x * y.expand_as(x)


class DWSepBlock(nn.Module):
    """Depthwise-separable conv block: DW3x3 -> BN -> act -> PW1x1 -> BN -> act,
    with an optional stride-2 downsample and a residual connection when shapes match."""
    def __init__(self, c_in: int, c_out: int, stride: int = 1, act=nn.SiLU):
        super().__init__()
        self.dw = nn.Conv2d(c_in, c_in, 3, stride=stride, padding=1, groups=c_in, bias=False)
        self.bn1 = nn.BatchNorm2d(c_in)
        self.pw = nn.Conv2d(c_in, c_out, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        self.act = act(inplace=True)
        self.use_res = (stride == 1 and c_in == c_out)

    def forward(self, x):
        identity = x
        x = self.act(self.bn1(self.dw(x)))
        x = self.bn2(self.pw(x))
        if self.use_res:
            x = x + identity
        return self.act(x)


class SpatialBackbone(nn.Module):
    """Lightweight from-scratch depthwise-separable stem + 3 stages. Exposes
    F1/F2/F3 (stage outputs, strides /4, /8, /16) for multi-scale fusion --
    deliberately NOT a pretrained off-the-shelf backbone."""
    def __init__(self, stem_ch=32, c1=48, c2=88, c3=160, depth=(1, 2, 2)):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, stem_ch, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(stem_ch),
            nn.SiLU(inplace=True),
        )
        self.stage1 = self._make_stage(stem_ch, c1, depth[0])
        self.stage2 = self._make_stage(c1, c2, depth[1])
        self.stage3 = self._make_stage(c2, c3, depth[2])
        self.out_channels = [c1, c2, c3]

    @staticmethod
    def _make_stage(c_in, c_out, n_extra):
        layers = [DWSepBlock(c_in, c_out, stride=2)]
        for _ in range(n_extra):
            layers.append(DWSepBlock(c_out, c_out, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)          # /2
        f1 = self.stage1(x)       # /4
        f2 = self.stage2(f1)      # /8
        f3 = self.stage3(f2)      # /16
        return f1, f2, f3


class MultiScaleSpatial(nn.Module):
    """Spatial representation Fs: projects F1/F2/F3 to a common width, aligns
    to F3's resolution, fuses with learned global softmax weights, then a
    depthwise residual refine."""
    def __init__(self, in_channels_list: List[int], out_channels: int):
        super().__init__()
        self.proj = nn.ModuleList([
            nn.Conv2d(c, out_channels, 1, bias=False) for c in in_channels_list
        ])
        self.bn = nn.ModuleList([nn.BatchNorm2d(out_channels) for _ in in_channels_list])
        self.scale_weight = nn.Parameter(torch.ones(len(in_channels_list)))
        self.refine = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, padding=1, groups=out_channels, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, feats: List[torch.Tensor]) -> torch.Tensor:
        target_hw = feats[-1].shape[-2:]
        aligned = [F.adaptive_avg_pool2d(bn(p(f)), target_hw)
                   for p, bn, f in zip(self.proj, self.bn, feats)]
        alpha = torch.softmax(self.scale_weight, dim=0)
        fused = sum(a * x for a, x in zip(alpha, aligned))
        return fused + self.refine(fused)


class MorphologyBranch(nn.Module):
    """Morphology-aware representation Fm. A fixed, parameter-free
    morphological-gradient operator (dilation - erosion via max-pool) applied
    to F3, then a learnable depthwise+pointwise transform, trained end-to-end."""
    def __init__(self, c_in: int, out_channels: int):
        super().__init__()
        self.pool = nn.MaxPool2d(3, stride=1, padding=1)
        self.dw = nn.Conv2d(c_in, c_in, 3, padding=1, groups=c_in, bias=False)
        self.bn1 = nn.BatchNorm2d(c_in)
        self.pw = nn.Conv2d(c_in, out_channels, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        dilation = self.pool(x)
        erosion = -self.pool(-x)
        grad = dilation - erosion          # morphological gradient, parameter-free
        x = self.act(self.bn1(self.dw(grad)))
        x = self.act(self.bn2(self.pw(x)))
        return x


class GatedFusion2(nn.Module):
    """Two-branch per-channel gated fusion (Spatial + Morphology). A
    per-channel, per-sample gate learned from both branches' pooled
    statistics -- gives the network the capacity to weight branches
    differently per output channel and per input image."""
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        hidden = max(channels // reduction, 8)
        self.gate = nn.Sequential(
            nn.Linear(channels * 2, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels * 2),
        )

    def forward(self, fs: torch.Tensor, fm: torch.Tensor):
        b, c = fs.shape[0], fs.shape[1]
        pooled = torch.cat([self.pool(fs).flatten(1), self.pool(fm).flatten(1)], dim=1)
        raw = self.gate(pooled).view(b, 2, c)
        weights = torch.softmax(raw, dim=1)          # per-channel, per-sample, sums to 1 over the 2 branches
        fused = (weights[:, 0, :].view(b, c, 1, 1) * fs +
                 weights[:, 1, :].view(b, c, 1, 1) * fm)
        return fused, weights


class SimpleAvgFusion(nn.Module):
    """Parameter-free 50/50 averaging fusion. Used ONLY in ablation config D
    ('remove Gated Fusion') so the model still has some way to combine
    Fs/Fm -- not part of the proposed architecture itself."""
    def forward(self, fs: torch.Tensor, fm: torch.Tensor):
        fused = 0.5 * fs + 0.5 * fm
        b, c = fs.shape[0], fs.shape[1]
        weights = torch.full((b, 2, c), 0.5, device=fs.device)
        return fused, weights


class SmallHead(nn.Module):
    """GAP -> small projection -> BatchNorm1d -> GELU -> Dropout -> Linear(4)."""
    def __init__(self, in_dim: int, hidden_dim: int, num_classes: int, dropout: float):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.bn = nn.BatchNorm1d(hidden_dim)
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.gap(x).flatten(1)
        emb = self.act(self.bn(self.fc1(x)))
        out = self.fc2(self.drop(emb))
        return out, emb

print("Shared building blocks ready (Spatial branch, Morphology branch, Gated Fusion — unchanged).")

In [ ]:
class MultiScaleContext(nn.Module):
    """Cheap multi-scale contextual extraction: channel-reduce with 1x1, three
    parallel DEPTHWISE 3x3 convs at dilation 1/2/3 (receptive fields 3/5/7),
    concat, expand back with 1x1. No full (non-depthwise, non-1x1) convs, per
    the lightweight-operations requirement."""
    def __init__(self, channels: int, reduction: int = 18):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.reduce = nn.Conv2d(channels, hidden, 1, bias=False)
        self.bn_reduce = nn.BatchNorm2d(hidden)
        self.dw1 = nn.Conv2d(hidden, hidden, 3, padding=1, dilation=1, groups=hidden, bias=False)
        self.dw2 = nn.Conv2d(hidden, hidden, 3, padding=2, dilation=2, groups=hidden, bias=False)
        self.dw3 = nn.Conv2d(hidden, hidden, 3, padding=3, dilation=3, groups=hidden, bias=False)
        self.bn_cat = nn.BatchNorm2d(hidden * 3)
        self.expand = nn.Conv2d(hidden * 3, channels, 1, bias=False)
        self.bn_expand = nn.BatchNorm2d(channels)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        r = self.act(self.bn_reduce(self.reduce(x)))
        m = torch.cat([self.dw1(r), self.dw2(r), self.dw3(r)], dim=1)
        m = self.act(self.bn_cat(m))
        return self.bn_expand(self.expand(m))


class CoordinateAttention(nn.Module):
    """Coordinate Attention (Hou et al., 2021), lightweight form: separately
    pools along H and W to preserve direction-specific information, encodes
    both with a shared 1x1 reduction, then splits back into an H-gate and a
    W-gate (both 1x1 projections, no full convolutions)."""
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        self.reduce = nn.Conv2d(channels, hidden, 1, bias=False)
        self.bn = nn.BatchNorm2d(hidden)
        self.act = nn.SiLU(inplace=True)
        self.conv_h = nn.Conv2d(hidden, channels, 1, bias=False)
        self.conv_w = nn.Conv2d(hidden, channels, 1, bias=False)

    def forward(self, x):
        b, c, h, w = x.shape
        x_h = self.pool_h(x)                          # (b,c,h,1) -- height-direction info
        x_w = self.pool_w(x).permute(0, 1, 3, 2)       # (b,c,w,1) -- width-direction info
        y = torch.cat([x_h, x_w], dim=2)               # (b,c,h+w,1)
        y = self.act(self.bn(self.reduce(y)))
        y_h, y_w = torch.split(y, [h, w], dim=2)
        y_w = y_w.permute(0, 1, 3, 2)                  # back to (b,hidden,1,w)
        g_h = torch.sigmoid(self.conv_h(y_h))          # (b,c,h,1)
        g_w = torch.sigmoid(self.conv_w(y_w))          # (b,c,1,w)
        return x * g_h * g_w


class MSCAECA(nn.Module):
    """Attention block of the proposed model. Multi-Scale Coordinate-ECA Attention. Fused feature ->
    parallel {MultiScaleContext, CoordinateAttention} -> ECA channel
    refinement -> residual. use_multiscale / use_coord / use_eca gate each
    stage independently for the internal-ablation study (F-I, section 16);
    a disabled stage is an identity pass-through, so this ONE class serves
    both the full module and every internal-ablation variant -- there is no
    separate ECA/SE/CBAM comparison code."""
    def __init__(self, channels: int, ms_reduction: int = 18, coord_reduction: int = 16,
                 use_multiscale: bool = True, use_coord: bool = True, use_eca: bool = True):
        super().__init__()
        self.use_multiscale = use_multiscale
        self.use_coord = use_coord
        self.use_eca = use_eca
        self.ms = MultiScaleContext(channels, reduction=ms_reduction) if use_multiscale else None
        self.coord = CoordinateAttention(channels, reduction=coord_reduction) if use_coord else None
        self.eca = ECA(channels) if use_eca else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        out = x
        if self.ms is not None:
            out = out + self.ms(out)
        if self.coord is not None:
            out = self.coord(out)
        if self.eca is not None:
            out = self.eca(out)
        return out + identity

print("MultiScaleContext, CoordinateAttention and MSCAECA ready.")

In [ ]:
# Channel configuration -- rep_width narrowed from SMGF-Net's 160 to 148 so that SMGF-Net + MS-CA-ECA
# stays within the 211,560-parameter budget of the reference SMGF-Net (plain ECA).
# Spatial branch / Morphology branch / Gated Fusion configs are otherwise IDENTICAL to the reference SMGF-Net.
SMGF_CONFIG = dict(stem_ch=32, c1=48, c2=88, c3=160, depth=(1, 2, 2),
                   rep_width=148, head_hidden=128)
ATTN_LABEL = "MS-CA-ECA"
MODEL_NAME = f"SMGF-Net + {ATTN_LABEL}"
MSCAECA_CONFIG = dict(ms_reduction=18, coord_reduction=16)
LESION_HEAD   = True    # auxiliary lesion-mask head (training signal only; classification path and pooling unchanged)
LESION_WEIGHT = 0.3     # weight of the auxiliary BCE + Dice loss
REFERENCE_SMGFNET_PARAMS = 211_560   # previous model (plain ECA), for the required parameter-diff report
PARAM_BUDGET = 211_560


class SMGFNet(nn.Module):
    """SMGF-Net + MS-CA-ECA -- the final proposed architecture, and the ONLY
    model class used throughout this notebook (full model + every ablation
    variant, selected via the flags below).

    Input MRI -> SpatialBackbone (stem + 3 depthwise-separable stages,
    exposing F1/F2/F3) -> {MultiScaleSpatial(Fs), MorphologyBranch(Fm)} ->
    GatedFusion2 -> MSCAECA -> SmallHead -> 4-class logits.

    use_spatial / use_morphology / use_gated_fusion: component ablation.
    ms_kwargs (use_multiscale / use_coord / use_eca): internal switches of the MS-CA-ECA block. Spatial branch, Morphology branch and Gated Fusion are NOT
    redesigned for any of these ablations -- only removed/replaced wholesale,
    exactly as the task brief specifies.
    """
    def __init__(self, num_classes: int = NUM_CLASSES, dropout: float = DROPOUT_RATE,
                 stem_ch: int = 32, c1: int = 48, c2: int = 88, c3: int = 160,
                 depth=(1, 2, 2), rep_width: int = 148, head_hidden: int = 128,
                 use_spatial: bool = True, use_morphology: bool = True,
                 use_gated_fusion: bool = True, use_lesion_head: bool = LESION_HEAD,
                 ms_kwargs: Optional[Dict[str, Any]] = None):
        super().__init__()
        assert use_spatial or use_morphology, "at least one of Spatial/Morphology must be active"
        self.use_spatial = use_spatial
        self.use_morphology = use_morphology
        self.use_gated_fusion = use_gated_fusion
        ms_kwargs = {**MSCAECA_CONFIG, **(ms_kwargs or {})}

        self.backbone = SpatialBackbone(stem_ch, c1, c2, c3, depth=depth)
        self.multiscale = MultiScaleSpatial(self.backbone.out_channels, rep_width) if use_spatial else None
        self.morphology = MorphologyBranch(c3, rep_width) if use_morphology else None

        if use_spatial and use_morphology:
            self.fusion = GatedFusion2(channels=rep_width) if use_gated_fusion else SimpleAvgFusion()
        else:
            self.fusion = None   # single-branch ablation (B/C): nothing to fuse

        self.attn = MSCAECA(rep_width, **ms_kwargs)   # attribute name kept for Grad-CAM++ / Score-CAM targeting

        # Lesion-guided auxiliary head: 1x1 conv on the attended map -> lesion logit map (H/16 x W/16). 149 parameters,
        # used only in the training loss; it does not feed the classifier, so inference cost and predictions are unchanged.
        self.lesion_head = nn.Conv2d(rep_width, 1, 1) if use_lesion_head else None

        self.head = SmallHead(rep_width, head_hidden, num_classes, dropout)

    def forward(self, x: torch.Tensor, return_all: bool = False) -> Dict[str, torch.Tensor]:
        f1, f2, f3 = self.backbone(x)
        fs = self.multiscale([f1, f2, f3]) if self.use_spatial else None
        fm = self.morphology(f3) if self.use_morphology else None

        weights = None
        if self.use_spatial and self.use_morphology:
            fused, weights = self.fusion(fs, fm)
        elif self.use_spatial:
            fused = fs
        else:
            fused = fm

        attn = self.attn(fused)
        logits, emb = self.head(attn)
        out = {"logits": logits, "emb": emb}
        if self.lesion_head is not None:
            out["lesion"] = self.lesion_head(attn)
        if return_all:
            out.update({"f1": f1, "f2": f2, "f3": f3, "spatial": fs, "morphology": fm,
                        "fusion_weights": weights, "fused": fused, "attn": attn})
        return out


def inference_param_count(model: nn.Module, trainable_only: bool = False) -> int:
    """Parameters of the deployed classifier: the training-only lesion head is not counted."""
    return sum(p.numel() for n, p in model.named_parameters()
               if not n.startswith("lesion_head.") and (p.requires_grad or not trainable_only))


def build_smgfnet(**kwargs) -> SMGFNet:
    """Builds SMGF-Net + MS-CA-ECA (or an ablation variant of it). Any of
    SMGFNet's constructor args can be overridden via kwargs, e.g.
    build_smgfnet(use_spatial=False) or
    build_smgfnet(ms_kwargs=dict(use_coord=False))."""
    cfg = {**SMGF_CONFIG}
    ms_override = kwargs.pop("ms_kwargs", None)
    cfg.update(kwargs)
    if ms_override is not None:
        cfg["ms_kwargs"] = ms_override
    model = SMGFNet(**cfg).to(DEVICE)
    return model


def param_breakdown(model: "SMGFNet") -> Dict[str, int]:
    def count(m):
        return 0 if m is None else sum(p.numel() for p in m.parameters() if p.requires_grad)
    parts = {
        "backbone (stem + 3 DW-sep stages)": count(model.backbone),
        "spatial branch (F1/F2/F3 -> Fs)": count(model.multiscale),
        "morphology branch (Fm)": count(model.morphology),
        "gated fusion (Fs, Fm -> fused)": count(model.fusion),
        f"{ATTN_LABEL} attention": count(model.attn),
        "classifier head": count(model.head),
    }
    parts["TOTAL"] = sum(v for k, v in parts.items())
    return parts


print("build_smgfnet() ready.")

In [ ]:
def expected_calibration_error(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15):
    """Returns (ece, bin_accs, bin_confs, bin_counts)."""
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies = (predictions == labels).astype(np.float32)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_accs, bin_confs, bin_counts = [], [], []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        count = mask.sum()
        if count == 0:
            bin_accs.append(0); bin_confs.append(0); bin_counts.append(0)
            continue
        bin_acc = accuracies[mask].mean()
        bin_conf = confidences[mask].mean()
        ece += (count / len(labels)) * abs(bin_acc - bin_conf)
        bin_accs.append(bin_acc); bin_confs.append(bin_conf); bin_counts.append(count)
    return ece, bin_accs, bin_confs, bin_counts


def compute_class_weights(samples: List[Dict], num_classes: int = NUM_CLASSES, power: float = 1.0) -> torch.Tensor:
    """Class weights proportional to (1/count)**power, normalized to mean 1.0. power=1 is plain inverse frequency (the old
    setting; it gives the rare no-tumor class ~2x the weight of meningioma, which matches the observed meningioma -> no-tumor
    errors), power=0.5 is a square-root compromise, power=0 is unweighted."""
    counts = np.zeros(num_classes)
    for s in samples:
        counts[s["label"]] += 1
    counts = np.maximum(counts, 1)
    weights = (counts.sum() / (num_classes * counts)) ** power
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)


def focal_weighted_ce(logits, targets, class_weights, label_smoothing=0.0, gamma=0.0):
    """Class-weighted, label-smoothed CE with an optional focal factor (1 - p_t)^gamma. gamma = 0 gives the same
    value as F.cross_entropy(..., weight=class_weights) (same weighted-mean normalisation)."""
    ce = F.cross_entropy(logits, targets, weight=class_weights, label_smoothing=label_smoothing, reduction="none")
    if gamma > 0:
        pt = F.softmax(logits, dim=1).gather(1, targets.view(-1, 1)).squeeze(1).detach()
        ce = ce * (1.0 - pt).clamp_min(0.0) ** gamma
    return ce.sum() / class_weights[targets].sum()


class CAWLPlus(nn.Module):
    def __init__(self, class_weights: torch.Tensor, num_classes: int = NUM_CLASSES,
                 emb_dim: int = 64, label_smoothing: float = 0.05,
                 target_confidence_cap: float = 0.85, lambda_max: float = 0.3,
                 anneal_epochs: int = 15, proto_momentum: float = 0.9,
                 proto_margin: float = 0.5, beta: float = 0.1,
                 proto_warmup_epochs: int = 8,
                 pair=(0, 1), pair_margin: float = 0.8, pair_weight: float = 0.0,
                 proto_min_lam: float = 0.0, focal_gamma: float = 0.0):
        super().__init__()
        self.focal_gamma = focal_gamma          # 0 = plain weighted CE
        self.register_buffer("class_weights", class_weights)
        self.label_smoothing = label_smoothing
        self.target_confidence_cap = target_confidence_cap
        self.lambda_max = lambda_max
        self.anneal_epochs = anneal_epochs
        self.current_epoch = 0

        self.num_classes = num_classes
        self.proto_momentum = proto_momentum
        self.proto_margin = proto_margin
        self.beta = beta
        # v3: extra margin between the glioma (0) and meningioma (1) prototypes, the pair that causes ~half of all test
        # errors; prototype terms are skipped on mixup batches where the dominant image has weight < proto_min_lam.
        self.pair = pair; self.pair_margin = pair_margin; self.pair_weight = pair_weight
        self.proto_min_lam = proto_min_lam
        self.proto_warmup_epochs = proto_warmup_epochs
        self.register_buffer("prototypes", torch.zeros(num_classes, emb_dim))
        self.register_buffer("proto_initialized", torch.zeros(num_classes, dtype=torch.bool))

    def _lambda_t(self):
        frac = min(self.current_epoch / max(self.anneal_epochs, 1), 1.0)
        return self.lambda_max * frac

    @torch.no_grad()
    def _update_prototypes(self, emb: torch.Tensor, targets: torch.Tensor):
        for c in range(self.num_classes):
            mask = targets == c
            if mask.sum() == 0:
                continue
            batch_mean = emb[mask].mean(dim=0)
            if not self.proto_initialized[c]:
                self.prototypes[c] = batch_mean
                self.proto_initialized[c] = True
            else:
                self.prototypes[c] = (self.proto_momentum * self.prototypes[c] +
                                       (1 - self.proto_momentum) * batch_mean)

    def _prototype_margin_loss(self, emb: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        if not self.proto_initialized.all():
            return torch.tensor(0.0, device=emb.device)
        protos = self.prototypes.to(emb.device)
        d = torch.cdist(emb, protos) ** 2
        d_pos = d.gather(1, targets.view(-1, 1)).squeeze(1)
        d_masked = d.clone()
        d_masked.scatter_(1, targets.view(-1, 1), float("inf"))
        d_neg = d_masked.min(dim=1).values
        return F.relu(d_pos - d_neg + self.proto_margin).mean()

    def _pair_margin_loss(self, emb: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """Hinge on squared distance: a sample of class a (or b) must be closer to its own prototype than to the
        other member of the pair by `pair_margin`. Only samples of the two pair classes contribute."""
        zero = emb.new_zeros(())
        if self.pair_weight <= 0 or not self.proto_initialized.all():
            return zero
        a, b = self.pair
        sel = (targets == a) | (targets == b)
        if not bool(sel.any()):
            return zero
        e, t = emb[sel], targets[sel]
        protos = self.prototypes.to(emb.device)
        d_a = ((e - protos[a]) ** 2).sum(1)
        d_b = ((e - protos[b]) ** 2).sum(1)
        d_pos = torch.where(t == a, d_a, d_b)
        d_oth = torch.where(t == a, d_b, d_a)
        return F.relu(d_pos - d_oth + self.pair_margin).mean()

    def forward(self, logits: torch.Tensor, targets: torch.Tensor,
                emb: Optional[torch.Tensor] = None) -> torch.Tensor:
        ce = focal_weighted_ce(logits, targets, self.class_weights, self.label_smoothing, self.focal_gamma)
        probs = F.softmax(logits, dim=1)
        conf, _ = probs.max(dim=1)
        pred = logits.argmax(dim=1)
        correct = (pred == targets).float()
        overconf = F.relu(conf - self.target_confidence_cap) * correct
        conf_penalty = overconf.mean()
        loss = ce + self._lambda_t() * conf_penalty

        if emb is not None and self.training:
            self._update_prototypes(emb.detach(), targets)   # EMA warms up silently from epoch 1
        if emb is not None and self.current_epoch >= self.proto_warmup_epochs:
            loss = loss + self.beta * self._prototype_margin_loss(emb, targets)  # gated
            loss = loss + self.pair_weight * self._pair_margin_loss(emb, targets)
        return loss

print("CAWLPlus ready (with proto_warmup_epochs gate).")

In [ ]:
def _file_hash(path: str) -> str:
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

def dedupe_splits(splits: Dict[str, List[Dict]],
                   priority: List[str] = ["test", "val", "train"]) -> Dict[str, List[Dict]]:
    hash_to_owner: Dict[str, Tuple[str, int]] = {}
    to_remove: Dict[str, set] = {name: set() for name in splits}
    n_cross = n_intra = 0

    for name in sorted(splits, key=lambda n: priority.index(n) if n in priority else 99):
        for idx, s in enumerate(splits[name]):
            h = _file_hash(s["path"])
            if h in hash_to_owner:
                owner_name, owner_idx = hash_to_owner[h]
                if owner_name == name:
                    n_intra += 1
                    if n_intra <= 5:
                        print(f"  [DUPLICATE within '{name}'] {splits[name][owner_idx]['path']}  ==  {s['path']}")
                else:
                    n_cross += 1
                    if n_cross <= 5:
                        print(f"  [LEAKAGE: kept in '{owner_name}', removed from '{name}'] {s['path']}")
                to_remove[name].add(idx)
            else:
                hash_to_owner[h] = (name, idx)

    cleaned = {name: [s for i, s in enumerate(samples) if i not in to_remove[name]]
               for name, samples in splits.items()}
    print(f"[LEAKAGE CHECK] {n_cross} cross-split leak(s) + {n_intra} intra-split "
          f"duplicate(s) auto-removed ({n_cross + n_intra} total).")
    for name in splits:
        removed = len(to_remove[name])
        if removed:
            print(f"  {name}: {len(splits[name])} -> {len(cleaned[name])} ({removed} removed)")
    return cleaned


def scan_dataset_generic(root: Path, folder_to_class: Dict[str, str], split_name: str) -> List[Dict[str, Any]]:
    samples = []
    for folder_name, cls in folder_to_class.items():
        cls_dir = root / folder_name
        if not cls_dir.exists():
            print(f"  [WARN] missing: {cls_dir}")
            continue
        n_before = len(samples)
        for p in cls_dir.rglob("*"):
            if p.is_file() and p.suffix.lower() in IMG_EXTS:
                samples.append({"path": str(p), "label": CLASS_TO_IDX[cls],
                                 "class_name": cls, "split": split_name})
        print(f"  {split_name}/{folder_name} -> {cls}: {len(samples) - n_before} images")
    print(f"{split_name}: {len(samples)} images across {NUM_CLASSES} classes")
    return samples

print("Scanner + leakage-check utilities ready.")

In [ ]:
def _find_brisc_root(base: Path = Path("/kaggle/input"), max_depth: int = 7) -> Optional[Path]:
    """Finds the folder that holds train/ and test/ with the four class folders, wherever Kaggle mounted the dataset."""
    default = base / "datasets/briscdataset/brisc2025/brisc2025/classification_task"
    if (default / "train").is_dir() and (default / "test").is_dir():
        return default
    if not base.exists():
        return None
    found = []
    for dirpath, dirnames, _ in os.walk(base):
        depth = len(Path(dirpath).relative_to(base).parts)
        if depth >= max_depth:
            dirnames[:] = []; continue
        names = set(dirnames)
        if {"train", "test"} <= names and (Path(dirpath) / "train" / "glioma").is_dir() and (Path(dirpath) / "test" / "glioma").is_dir():
            found.append(Path(dirpath))
        if names & {"glioma", "meningioma", "pituitary", "no_tumor"}:
            dirnames[:] = []                      # do not walk into class folders full of images
    return sorted(found, key=lambda p: ("classification_task" not in p.parts, len(p.parts)))[0] if found else None

BRISC_ROOT = _find_brisc_root()
if BRISC_ROOT is None:
    _seen = [str(p) for p in sorted(Path("/kaggle/input").glob("*"))][:20] + [str(p) for p in sorted(Path("/kaggle/input").glob("*/*"))][:20]
    raise RuntimeError("BRISC2025 not found under /kaggle/input. Attach it to this notebook (Add Input -> search 'brisc2025'). "
                       f"Currently mounted: {_seen}")
BRISC_TRAIN_ROOT = BRISC_ROOT / "train"
BRISC_TEST_ROOT  = BRISC_ROOT / "test"
print(f"BRISC2025 root: {BRISC_ROOT}")
print(f"BRISC2025 train exists: {BRISC_TRAIN_ROOT.exists()} -> {BRISC_TRAIN_ROOT}")
print(f"BRISC2025 test  exists: {BRISC_TEST_ROOT.exists()} -> {BRISC_TEST_ROOT}")

SEG_ROOT = BRISC_ROOT.parent / "segmentation_task"
MASK_INDEX.clear()
for _split in ("train", "test"):
    _d = SEG_ROOT / _split / "masks"
    if _d.is_dir():
        for _p in _d.iterdir():
            if _p.suffix.lower() in IMG_EXTS:
                MASK_INDEX[(_split, _norm_stem(_p.stem))] = str(_p)
print(f"Lesion masks indexed: {len(MASK_INDEX)} from {SEG_ROOT}")

BRISC_FOLDER_TO_CLASS = {
    "glioma": "glioma", "meningioma": "meningioma",
    "no_tumor": "notumor", "pituitary": "pituitary",
}

In [ ]:
# ===== Near-duplicate audit and cleaning (runs BEFORE any training) =====
# md5 dedupe only removes byte-identical files. Re-encoded copies, light crops and neighbouring slices of one scan survive it
# and can sit on both sides of the train/test line. Here every image is reduced to a 32x32 zero-mean unit-norm thumbnail;
# the dot product of two thumbnails is their normalised correlation (1.0 = identical).
#   1. TRAIN images with a near-identical TEST image (similarity >= NEAR_DUP_THRESHOLD) are dropped from train.
#      The test set is never changed, so the benchmark stays comparable.
#   2. Groups of near-identical TRAIN images are collapsed to one representative (the lexicographically first path), so the
#      random train/validation split cannot put copies on both sides.
#   3. A group whose members carry different labels is ambiguous and is dropped entirely.
# run_experiment() calls near_dup_clean() on every run, so seeds and grid runs all train on the same cleaned pool.
NEAR_DUP_CLEAN     = True
NEAR_DUP_THRESHOLD = 0.98     # normalised correlation; the band table printed below shows how many images sit near this value

from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

_THUMBS: Dict[str, np.ndarray] = {}
def _thumb_vec(path: str, n: int = 32) -> np.ndarray:
    v = _THUMBS.get(path)
    if v is None:
        g = cv2.resize(read_gray(path), (n, n), interpolation=cv2.INTER_AREA).astype(np.float32)
        g -= g.mean()
        v = (g / (np.linalg.norm(g) + 1e-8)).ravel()
        _THUMBS[path] = v
    return v

_NEAR_DUP_BANDS = (0.0, 0.90, 0.95, 0.98, 0.995, 1.01)

def near_dup_clean(train_s, test_s, thr=None, verbose=True):
    thr = NEAR_DUP_THRESHOLD if thr is None else thr
    Tr = np.stack([_thumb_vec(s["path"]) for s in train_s])
    Te = np.stack([_thumb_vec(s["path"]) for s in test_s])
    sim_te = (Tr @ Te.T).max(1)                                   # best test match for every train image
    drop_test = sim_te >= thr
    keep = np.where(~drop_test)[0]
    S = Tr[keep] @ Tr[keep].T
    np.fill_diagonal(S, 0.0)
    ncomp, comp = connected_components(csr_matrix(S >= thr), directed=False)
    labels = np.array([train_s[i]["label"] for i in keep]); paths = np.array([train_s[i]["path"] for i in keep])
    drop_dup, drop_mixed, kept_reps = set(), set(), 0
    for g in range(ncomp):
        members = np.where(comp == g)[0]
        if len(members) == 1:
            continue
        if len(set(labels[members])) > 1:
            drop_mixed.update(members.tolist()); continue
        rep = members[np.argmin(paths[members])]
        drop_dup.update(m for m in members.tolist() if m != rep); kept_reps += 1
    kept_local = [j for j in range(len(keep)) if j not in drop_dup and j not in drop_mixed]
    cleaned = [train_s[keep[j]] for j in kept_local]
    report = {"threshold": thr, "n_train_in": len(train_s), "n_train_out": len(cleaned),
              "removed_matching_test": int(drop_test.sum()), "removed_train_duplicates": len(drop_dup),
              "removed_mixed_label_groups_images": len(drop_mixed), "duplicate_groups_collapsed": kept_reps}
    removed = [train_s[i]["path"] for i in np.where(drop_test)[0]] + [paths[j] for j in sorted(drop_dup | drop_mixed)]
    report["removed_paths"] = removed
    print(f"[NEAR-DUP] threshold {thr}: train {len(train_s)} -> {len(cleaned)}  "
          f"(vs test: -{report['removed_matching_test']}, duplicate groups: -{report['removed_train_duplicates']}, "
          f"mixed-label: -{report['removed_mixed_label_groups_images']})")
    if verbose:
        rows = [{"similarity": f"[{lo:.3f}, {min(hi, 1.0):.3f}{']' if hi > 1 else ')'}",
                 "train images whose closest test image falls here": int(((sim_te >= lo) & (sim_te < hi)).sum())}
                for lo, hi in zip(_NEAR_DUP_BANDS[:-1], _NEAR_DUP_BANDS[1:])]
        print(pd.DataFrame(rows).to_string(index=False))
        print(f"closest train->test similarity: median {np.median(sim_te):.3f}  p99 {np.quantile(sim_te, .99):.3f}  max {sim_te.max():.3f}")
    return cleaned, report


# ---- leak-safe train/validation split -------------------------------------------------------------------------------
# near_dup_clean() only collapses copies at >= NEAR_DUP_THRESHOLD. Neighbouring slices of one scan sit lower (about 0.85-0.98),
# so a random 85/15 split still puts them on both sides; that is why validation reached 98.5-99 % while test stayed at 93-96 %.
# Here images are linked at a lower threshold, connected components become groups, and whole groups go to one side.
VAL_GROUP_THRESHOLDS = (0.90, 0.93, 0.95, 0.98)   # lowest value whose largest group stays <= 3 % of the pool is used
VAL_GROUP_MAX_FRAC   = 0.03

def grouped_train_val_split(samples, val_frac=0.15, seed=None, verbose=True):
    from sklearn.model_selection import StratifiedGroupKFold
    seed = SEED if seed is None else seed
    X = np.stack([_thumb_vec(s["path"]) for s in samples]); y = np.array([s["label"] for s in samples])
    Sm = X @ X.T; np.fill_diagonal(Sm, 0.0)
    for thr in VAL_GROUP_THRESHOLDS:
        ncomp, comp = connected_components(csr_matrix(Sm >= thr), directed=False)
        biggest = np.bincount(comp).max() / len(samples)
        if biggest <= VAL_GROUP_MAX_FRAC: break
    del Sm
    n_splits = max(2, round(1 / val_frac))
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    tr_idx, va_idx = next(sgkf.split(X, y, groups=comp))
    if verbose:
        print(f"[GROUP SPLIT] link threshold {thr}: {ncomp} groups, largest {biggest:.1%} of the pool; "
              f"train {len(tr_idx)} / val {len(va_idx)}; val class counts {np.bincount(y[va_idx], minlength=NUM_CLASSES).tolist()}")
    return [samples[i] for i in tr_idx], [samples[i] for i in va_idx]

# ---- the audit, before any training ----
_pre_train = scan_dataset_generic(BRISC_TRAIN_ROOT, BRISC_FOLDER_TO_CLASS, "train")
_pre_test  = scan_dataset_generic(BRISC_TEST_ROOT,  BRISC_FOLDER_TO_CLASS, "test")
_pre_clean, near_dup_report = near_dup_clean(_pre_train, _pre_test, verbose=True)
pd.Series(near_dup_report["removed_paths"], name="removed_path").to_csv(OUT_DIR / "near_duplicate_removed.csv", index=False)
print(f"removed paths written to {OUT_DIR / 'near_duplicate_removed.csv'}")

In [ ]:
TOTAL_EPOCHS  = 60       # was 50. Val accuracy only reached ~80% at epoch 13 and was still creeping up under the decaying LR.
# AdamW decays each weight by lr*wd per step. With lr=1.8e-4 and wd=3e-4 that is ~5e-8 per step, i.e. no
# regularisation at all, while the clean train/val gap was ~99.5% vs ~95%. 5e-2 is the usual AdamW range.
WEIGHT_DECAY  = 5e-2
GRAD_CLIP     = 1.0
PEAK_LR       = 5e-4     # was 1.8e-4 (and 4e-4 before that). The least certain change here; if the curves turn unstable, try 3e-4.
MIN_LR        = 1e-6
# Sweep grids used by the optional sweep cell after the A/B cell (RUN_SWEEP is False by default: GPU quota)
SWEEP_WEIGHT_DECAY = [1e-4, 1e-3, 1e-2, 5e-2, 1e-1]    # 1e-4 .. 1e-1
SWEEP_PEAK_LR      = [1e-4, 3e-4, 5e-4, 1e-3]          # 1e-4 .. 1e-3
# Regularisation knobs read by run_experiment
MIXUP_ALPHA  = 0.4     # only used if a profile sets mixup=MIXUP_ALPHA (the 93.27 % run); the final profile uses 0.2
CUTMIX_ALPHA = 1.0     # new: each batch uses CutMix or Mixup with probability 0.5 each
CUTMIX_PROB  = 0.5     # share of batches that use CutMix instead of Mixup
FOCAL_GAMMA  = 2.0     # new: focal factor (1 - p_t)^gamma inside the CAWL+ cross-entropy

WARMUP_EPOCHS_V2     = 10   # was 5 (15 before that); a 15-epoch ramp from 3.6e-6 spent 30% of the run below useful LR
WARMUP_START_DIV_V2  = 50

FREEZE_BACKBONE_EPOCHS = 3

# Fine-grained early logging: for the first FINEGRAIN_EPOCHS epochs, evaluate
# train(clean)/val this many times DURING the epoch (not just once at the end).
# A frozen-backbone linear-probe head converges fast -- within a single epoch it can
# already climb from chance-level (~25% for 4 classes) most of the way to its plateau.
# Logging only once per epoch was hiding that climb and making it look like val
# "jumped" straight to a high number with no visible ramp. This makes the ramp visible.
FINEGRAIN_EPOCHS   = 2
FINEGRAIN_SUBSTEPS = 5

def lr_at_epoch_v2(epoch: int) -> float:
    warmup_start_lr = PEAK_LR / WARMUP_START_DIV_V2
    if epoch <= WARMUP_EPOCHS_V2:
        frac = epoch / WARMUP_EPOCHS_V2
        return warmup_start_lr + frac * (PEAK_LR - warmup_start_lr)
    remaining_epochs = TOTAL_EPOCHS - WARMUP_EPOCHS_V2
    progress = min((epoch - WARMUP_EPOCHS_V2) / max(remaining_epochs, 1), 1.0)
    return MIN_LR + 0.5 * (PEAK_LR - MIN_LR) * (1 + math.cos(math.pi * progress))


def train_one_epoch_v2(model, loader, optimizer, criterion: CAWLPlus,
                        scaler: Optional[GradScaler] = None):
    model.train()
    total_loss = correct = total = 0
    use_amp = scaler is not None
    for batch in loader:
        img   = batch["image"].to(DEVICE, non_blocking=True)
        label = batch["label"].to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            with autocast():
                out  = model(img)
                loss = criterion(out["logits"], label, out["emb"])
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            out  = model(img)
            loss = criterion(out["logits"], label, out["emb"])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        pred = out["logits"].argmax(1)
        correct += (pred == label).sum().item()
        total   += label.size(0)
        total_loss += loss.item() * label.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_v2(model, loader, criterion: CAWLPlus, device: str = DEVICE) -> Dict[str, Any]:
    model.eval()
    _crit_was_training = criterion.training
    criterion.eval()   # evaluation must not update the loss's class prototypes (it did before: val/test labels leaked into them)
    all_preds, all_labels, all_probs, total_loss = [], [], [], 0.0
    for batch in loader:
        img   = batch["image"].to(device, non_blocking=True)
        label = batch["label"].to(device, non_blocking=True)
        out   = model(img)
        loss  = criterion(out["logits"], label, out["emb"])
        total_loss += loss.item() * label.size(0)
        probs = torch.softmax(out["logits"], dim=1).cpu().numpy()
        all_probs.extend(probs.tolist())
        all_preds.extend(out["logits"].argmax(1).cpu().tolist())
        all_labels.extend(label.cpu().tolist())

    criterion.train(_crit_was_training)
    n = len(all_labels)
    y_true = np.array(all_labels); y_pred = np.array(all_preds); y_prob = np.array(all_probs)
    try:
        auc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except Exception:
        auc = float("nan")
    cm = confusion_matrix(y_true, y_pred)
    ece, _, _, _ = expected_calibration_error(y_prob, y_true)

    return {
        "loss": total_loss / n,
        "acc": accuracy_score(y_true, y_pred),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "precision_mac": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_mac": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_mac": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "auc": auc,
        "mcc": matthews_corrcoef(y_true, y_pred),
        "kappa": cohen_kappa_score(y_true, y_pred),
        "ece": ece,
        "cm": cm,
        "report": classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0),
    }


def plot_confusion_matrix(cm: np.ndarray, title: str = "Confusion Matrix", save_path: Optional[str] = None):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right"); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.colorbar(im, ax=ax); plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

# Fine-grained ("chunked") epoch trainer -- identical math to train_one_epoch_v2,
# but pauses n_chunks-1 times during the epoch to log a clean train/val snapshot at
# that fractional-epoch point. Only used for the first FINEGRAIN_EPOCHS epochs.
def train_one_epoch_chunked_v2(model, loader, optimizer, criterion: CAWLPlus,
                                scaler: Optional[GradScaler], n_chunks: int, epoch: int,
                                train_eval_loader, val_loader, history: Dict[str, list]):
    model.train()
    use_amp = scaler is not None
    batches = list(loader)
    n_batches = len(batches)
    chunk_size = max(1, n_batches // n_chunks)
    total_loss = correct = total = 0

    for i, batch in enumerate(batches):
        img   = batch["image"].to(DEVICE, non_blocking=True)
        label = batch["label"].to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            with autocast():
                out  = model(img)
                loss = criterion(out["logits"], label, out["emb"])
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            out  = model(img)
            loss = criterion(out["logits"], label, out["emb"])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        pred = out["logits"].argmax(1)
        correct += (pred == label).sum().item()
        total   += label.size(0)
        total_loss += loss.item() * label.size(0)

        is_chunk_boundary = ((i + 1) % chunk_size == 0) and (i + 1) < n_batches
        if is_chunk_boundary:
            frac_epoch = (epoch - 1) + (i + 1) / n_batches
            train_clean = evaluate_v2(model, train_eval_loader, criterion)
            val_metrics = evaluate_v2(model, val_loader, criterion)
            print(f"  epoch {frac_epoch:06.2f} (substep)  "
                  f"train_acc(clean)={train_clean['acc']:.4f}  "
                  f"val_acc={val_metrics['acc']:.4f}  val_auc={val_metrics['auc']:.4f}")
            history["epoch"].append(frac_epoch)
            history["train_loss"].append(train_clean["loss"])
            history["train_acc"].append(train_clean["acc"])
            history["train_loss_aug"].append(total_loss / total)
            history["train_acc_aug"].append(correct / total)
            history["val_loss"].append(val_metrics["loss"])
            history["val_acc"].append(val_metrics["acc"])
            history["val_auc"].append(val_metrics["auc"])
            history["val_ece"].append(val_metrics["ece"])
            history["lr"].append(optimizer.param_groups[0]["lr"])
            model.train()

    return total_loss / total, correct / total


print(f"Schedule ready: {TOTAL_EPOCHS} epochs, {WARMUP_EPOCHS_V2}-epoch gentle warmup "
      f"(start={PEAK_LR/WARMUP_START_DIV_V2:.1e}), peak_lr={PEAK_LR:.1e}, AMP={USE_AMP}, "
      f"early_stop_patience={EARLY_STOP_PATIENCE}, freeze_backbone_epochs={FREEZE_BACKBONE_EPOCHS}, "
      f"finegrain_epochs={FINEGRAIN_EPOCHS}x{FINEGRAIN_SUBSTEPS}")

In [ ]:
# ===== Validation-only headline mode and saved predictions (review: frozen recipe, TTA decision) ======
# Nothing in run_experiment() is chosen with test labels. The only inference-mode decision (flip-TTA vs single pass)
# is made by the rule below on the VALIDATION split, before the test split is scored. Rule, fixed in advance:
#   use flip-TTA if it lowers validation NLL AND does not lower validation accuracy; otherwise single-pass.
@torch.no_grad()
def probs_single_and_tta(model, loader):
    model.eval(); ys, p1s, p2s = [], [], []
    for b in loader:
        x = b["image"].to(DEVICE, non_blocking=True)
        p1 = torch.softmax(model(x)["logits"], 1)
        p2 = torch.softmax(model(torch.flip(x, dims=[3]))["logits"], 1)
        ys.append(b["label"].numpy()); p1s.append(p1.cpu().numpy()); p2s.append(p2.cpu().numpy())
    y = np.concatenate(ys); p1 = np.concatenate(p1s); p2 = np.concatenate(p2s)
    return y, p1, (p1 + p2) / 2


def acc_and_nll(y, p):
    return (float((p.argmax(1) == y).mean()),
            float(-np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1.0)).mean()))


def choose_headline_mode(y_val, p_single, p_tta):
    a1, n1 = acc_and_nll(y_val, p_single); a2, n2 = acc_and_nll(y_val, p_tta)
    mode = "flip-TTA" if (n2 < n1 and a2 >= a1) else "single-pass"
    return mode, {"val_acc_single": a1, "val_nll_single": n1, "val_acc_tta": a2, "val_nll_tta": n2}


print("Validation-only headline-mode helpers ready.")

In [ ]:
import re as _re
from torch.utils.data import WeightedRandomSampler
PLANE_BALANCE = False      # True: inside every class, the three planes (ax/co/sa) are sampled equally often; class totals stay unchanged
SOURCE_BALANCE = False   # True: reweight so that the image source (512x512 file vs other size) does not predict the class
SOURCE_W_CLIP  = 4.0     # cap on a single image weight
SELECT_BY     = "auc"      # "auc" (old) or "val_loss": what picks the checkpoint and drives early stopping
def _plane_of(path: str) -> str:
    m = _re.search(r"_(ax|co|sa)_", str(path)); return m.group(1) if m else "na"
def _sel(m: Dict[str, Any]) -> float:
    return -m["loss"] if SELECT_BY == "val_loss" else m["auc"]

TRAIN_EVAL_N = 1500       # images used for the per-epoch 'clean train accuracy' (was len(val): 282-684 images, where 100 % is only 282-684 correct)
CKPT_MIN_EPOCH = 0       # no checkpoint is taken and no early-stop counting happens before this epoch (the last 'base6' run stopped at 35 and kept epoch 19)
VAL_FRAC = 0.15           # share of the cleaned pool held out (by near-duplicate group) for monitoring / early stopping
GROUPED_VAL_SPLIT = True   # validation = whole near-duplicate groups, so val accuracy is no longer inflated by neighbours of training images

class ModelEMA:
    """Exponential moving average of the model's weights."""
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model: nn.Module):
        model.load_state_dict(self.shadow, strict=True)


def mixup_batch(img: torch.Tensor, label: torch.Tensor, alpha: float = 0.2):
    """Standard Mixup (Zhang et al., 2018). Returns (mixed, y_a, y_b, lam, idx, box); box is None for Mixup."""
    if alpha <= 0:
        return img, label, label, 1.0, None, None
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(img.size(0), device=img.device)
    mixed = lam * img + (1 - lam) * img[idx]
    return mixed, label, label[idx], lam, idx, None


def cutmix_batch(img: torch.Tensor, label: torch.Tensor, alpha: float = 1.0,
                 mask: Optional[torch.Tensor] = None, max_cover: float = 0.10, tries: int = 10):
    """CutMix (Yun et al., 2019): paste a random box from a shuffled copy of the batch; lam = kept-area fraction.
    Lesion-safe: with a mask given, the box is redrawn (up to `tries` times) until it covers at most `max_cover` of the
    lesion area of the images it is pasted over; if no box qualifies the batch falls back to Mixup, so a small lesion is
    never overwritten while the label still credits the original image.
    Returns (mixed, y_a, y_b, lam, idx, box)."""
    H, W = img.shape[-2:]
    for _ in range(tries if mask is not None else 1):
        lam = float(np.random.beta(alpha, alpha))
        r = math.sqrt(1.0 - lam)
        ch, cw = int(H * r), int(W * r)
        cy, cx = np.random.randint(H), np.random.randint(W)
        y1, y2 = max(cy - ch // 2, 0), min(cy + ch // 2, H)
        x1, x2 = max(cx - cw // 2, 0), min(cx + cw // 2, W)
        if mask is None:
            break
        tot = float(mask.sum())
        if tot <= 0 or float(mask[:, :, y1:y2, x1:x2].sum()) / tot <= max_cover:
            break
    else:
        return mixup_batch(img, label, alpha=0.2)                 # no lesion-safe box found
    idx = torch.randperm(img.size(0), device=img.device)
    mixed = img.clone()
    mixed[:, :, y1:y2, x1:x2] = img[idx][:, :, y1:y2, x1:x2]
    lam = 1.0 - (y2 - y1) * (x2 - x1) / float(H * W)
    return mixed, label, label[idx], lam, idx, (y1, y2, x1, x2)


def mix_mask(mask: torch.Tensor, valid: torch.Tensor, lam: float, idx, box):
    """Applies the same Mixup / CutMix to the lesion mask, so the target matches the mixed image."""
    if idx is None:
        return mask, valid
    if box is None:
        m = lam * mask + (1 - lam) * mask[idx]
    else:
        y1, y2, x1, x2 = box
        m = mask.clone(); m[:, :, y1:y2, x1:x2] = mask[idx][:, :, y1:y2, x1:x2]
    return m, valid * valid[idx]


def lesion_aux_loss(lesion_logits: torch.Tensor, mask: torch.Tensor, valid: torch.Tensor) -> torch.Tensor:
    """BCE + soft Dice between the lesion logit map and the mask average-pooled to the same resolution, averaged
    over samples with a valid mask."""
    z = lesion_logits.float()
    t = F.adaptive_avg_pool2d(mask.float(), z.shape[-2:])
    w = valid.view(-1).float()
    if float(w.sum()) < 1.0:
        return z.sum() * 0.0
    bce = F.binary_cross_entropy_with_logits(z, t, reduction="none").mean(dim=(1, 2, 3))
    p = torch.sigmoid(z)
    inter = (p * t).sum(dim=(1, 2, 3)); den = p.sum(dim=(1, 2, 3)) + t.sum(dim=(1, 2, 3))
    dice = 1.0 - (2.0 * inter + 1.0) / (den + 1.0)
    return ((bce + dice) * w).sum() / w.sum()


def mixup_cawl_loss(criterion: CAWLPlus, logits: torch.Tensor, emb: torch.Tensor,
                     y_a: torch.Tensor, y_b: torch.Tensor, lam: float) -> torch.Tensor:
    """Mixup-safe re-implementation of CAWLPlus.forward() (§9): computes CE +
    confidence-penalty for both mixed labels, but touches the prototype
    buffer only once per batch (dominant label), to avoid the in-place
    prototype-mutation issue of calling criterion(...) twice."""
    ce_a = focal_weighted_ce(logits, y_a, criterion.class_weights, criterion.label_smoothing, criterion.focal_gamma)
    ce_b = focal_weighted_ce(logits, y_b, criterion.class_weights, criterion.label_smoothing, criterion.focal_gamma)
    ce = lam * ce_a + (1 - lam) * ce_b

    probs = F.softmax(logits, dim=1)
    conf, _ = probs.max(dim=1)
    pred = logits.argmax(dim=1)
    correct = lam * (pred == y_a).float() + (1 - lam) * (pred == y_b).float()
    conf_penalty = (F.relu(conf - criterion.target_confidence_cap) * correct).mean()
    loss = ce + criterion._lambda_t() * conf_penalty

    y_dom = y_a if lam >= 0.5 else y_b
    if max(lam, 1.0 - lam) >= criterion.proto_min_lam:      # prototype terms only on mostly-one-image batches
        if criterion.training:
            criterion._update_prototypes(emb.detach(), y_dom)
        if criterion.current_epoch >= criterion.proto_warmup_epochs:
            loss = loss + criterion.beta * criterion._prototype_margin_loss(emb, y_dom) \
                        + criterion.pair_weight * criterion._pair_margin_loss(emb, y_dom)
    return loss


def train_one_epoch_mixup(model, loader, optimizer, criterion: CAWLPlus,
                           scaler: Optional[GradScaler] = None,
                           ema: Optional[ModelEMA] = None,
                           mixup_alpha: float = 0.2, cutmix_alpha: float = 0.0):
    model.train()
    total_loss = correct = total = 0
    use_amp = scaler is not None
    for batch in loader:
        img   = batch["image"].to(DEVICE, non_blocking=True)
        label = batch["label"].to(DEVICE, non_blocking=True)
        use_lesion = LESION_HEAD and LESION_WEIGHT > 0 and getattr(model, "lesion_head", None) is not None
        if use_lesion:
            mask_b = batch["mask"].to(DEVICE, non_blocking=True)
            valid_b = batch["mask_valid"].to(DEVICE, non_blocking=True)
        if cutmix_alpha > 0 and np.random.rand() < CUTMIX_PROB:     # per batch: CutMix or Mixup, equal probability
            img_m, y_a, y_b, lam, _idx, _box = cutmix_batch(img, label, alpha=cutmix_alpha, mask=mask_b if use_lesion else None)
        else:
            img_m, y_a, y_b, lam, _idx, _box = mixup_batch(img, label, alpha=mixup_alpha)
        if use_lesion:
            mask_m, mvalid_m = mix_mask(mask_b, valid_b, lam, _idx, _box)
        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            with autocast():
                out  = model(img_m)
                loss = mixup_cawl_loss(criterion, out["logits"], out["emb"], y_a, y_b, lam)
                if use_lesion: loss = loss + LESION_WEIGHT * lesion_aux_loss(out["lesion"], mask_m, mvalid_m)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            out  = model(img_m)
            loss = mixup_cawl_loss(criterion, out["logits"], out["emb"], y_a, y_b, lam)
            if use_lesion: loss = loss + LESION_WEIGHT * lesion_aux_loss(out["lesion"], mask_m, mvalid_m)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        if ema is not None:
            ema.update(model)

        pred = out["logits"].argmax(1)
        correct += (lam * (pred == y_a).float() + (1 - lam) * (pred == y_b).float()).sum().item()
        total   += label.size(0)
        total_loss += loss.item() * label.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_v2_tta(model, loader, criterion: CAWLPlus, device: str = DEVICE) -> Dict[str, Any]:
    """Same metric suite as evaluate_v2 (§12), but averages softmax
    probabilities over the original image and its horizontal flip
    (test-time augmentation) before taking the argmax."""
    model.eval()
    _crit_was_training = criterion.training
    criterion.eval()
    all_preds, all_labels, all_probs, total_loss = [], [], [], 0.0
    for batch in loader:
        img   = batch["image"].to(device, non_blocking=True)
        label = batch["label"].to(device, non_blocking=True)
        out1  = model(img)
        out2  = model(torch.flip(img, dims=[3]))
        probs = (torch.softmax(out1["logits"], dim=1) + torch.softmax(out2["logits"], dim=1)) / 2
        loss  = criterion(out1["logits"], label, out1["emb"])
        total_loss += loss.item() * label.size(0)
        all_probs.extend(probs.cpu().numpy().tolist())
        all_preds.extend(probs.argmax(1).cpu().tolist())
        all_labels.extend(label.cpu().tolist())

    criterion.train(_crit_was_training)
    n = len(all_labels)
    y_true = np.array(all_labels); y_pred = np.array(all_preds); y_prob = np.array(all_probs)
    try:
        auc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except Exception:
        auc = float("nan")
    cm = confusion_matrix(y_true, y_pred)
    ece, _, _, _ = expected_calibration_error(y_prob, y_true)
    return {
        "loss": total_loss / n,
        "acc": accuracy_score(y_true, y_pred),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "precision_mac": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_mac": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_mac": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "auc": auc,
        "mcc": matthews_corrcoef(y_true, y_pred),
        "kappa": cohen_kappa_score(y_true, y_pred),
        "ece": ece,
        "cm": cm,
        "report": classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0),
    }


def run_experiment(dataset_name: str, train_root: Path, test_root: Path,
                    folder_to_class: Dict[str, str],
                    model_builder=None,
                    total_epochs: Optional[int] = None,
                    dropout: Optional[float] = None,
                    weight_decay: Optional[float] = None,
                    mixup_alpha: Optional[float] = None,
                    cutmix_alpha: Optional[float] = None,
                    ema_decay: float = 0.999,
                    model_kwargs: Optional[Dict[str, Any]] = None,
                    loss_kwargs: Optional[Dict[str, Any]] = None,
                    epoch_callback=None) -> Dict[str, Any]:
    """Trains a model built by `model_builder(dropout=..., **model_kwargs)`
    (defaults to build_smgfnet -- the full SMGF-Net + MS-CA-ECA model)
    with the SAME dataset loading / splitting / leakage-check / preprocessing
    / augmentation / optimizer / LR-schedule / CAWL+ loss / Mixup / EMA /
    flip-TTA pipeline every time. Used for the full model (§14) AND every
    component/internal ablation variant (§15/§16) -- this identical function
    call is what keeps the ablation protocol fair (section 5 of the task
    brief): same splits, same preprocessing, same augmentation, same image
    size, same batch size, same optimizer, same LR schedule, same epochs,
    same loss, same early stopping, same seed."""
    model_builder = model_builder or build_smgfnet
    mixup_alpha = MIXUP_ALPHA if mixup_alpha is None else mixup_alpha
    cutmix_alpha = CUTMIX_ALPHA if cutmix_alpha is None else cutmix_alpha
    total_epochs = TOTAL_EPOCHS if total_epochs is None else total_epochs
    dropout = 0.4 if dropout is None else dropout
    weight_decay = WEIGHT_DECAY if weight_decay is None else weight_decay
    model_kwargs = model_kwargs or {}
    loss_kwargs = dict(loss_kwargs or {})
    loss_kwargs.setdefault("focal_gamma", FOCAL_GAMMA)
    cw_power = loss_kwargs.pop("cw_power", 1.0)      # class-weight exponent, see compute_class_weights

    print("\n" + "#" * 65)
    print(f"# EXPERIMENT: {dataset_name}  "
          f"[dropout={dropout} weight_decay={weight_decay:.2e} mixup_alpha={mixup_alpha} cutmix_alpha={cutmix_alpha} focal_gamma={loss_kwargs.get('focal_gamma')} peak_lr={PEAK_LR:.1e} "
          f"ema_decay={ema_decay} epochs={total_epochs} model_kwargs={model_kwargs}]")
    print("#" * 65)

    all_train = scan_dataset_generic(train_root, folder_to_class, "train")
    test_s    = scan_dataset_generic(test_root, folder_to_class, "test")
    if len(all_train) == 0 or len(test_s) == 0:
        raise RuntimeError(f"[{dataset_name}] No images found — check paths.")
    if NEAR_DUP_CLEAN:
        all_train, _ = near_dup_clean(all_train, test_s, verbose=False)   # §11b; test set is never modified
    if GROUPED_VAL_SPLIT:
        train_s, val_s = grouped_train_val_split(all_train, VAL_FRAC, SEED)
    else:
        train_s, val_s = train_test_split(
            all_train, test_size=VAL_FRAC, random_state=SEED,
            stratify=[s["label"] for s in all_train])

    splits = dedupe_splits({"train": train_s, "val": val_s, "test": test_s})
    train_s, val_s, test_s = splits["train"], splits["val"], splits["test"]

    if LESION_HEAD:
        _tum = [_s for _s in train_s if _s["class_name"] != "notumor"]
        _cov = sum(mask_path_of(_s) is not None for _s in _tum) / max(len(_tum), 1)
        print(f"  lesion masks found for {_cov:.1%} of the {len(_tum)} tumour images in the training split")
        if _cov < 0.5:
            raise RuntimeError("Fewer than half of the tumour training images have a lesion mask. Check MASK_INDEX / file names "
                               "in the segmentation_task folder, or set LESION_HEAD = False.")
    train_ds = BrainTumorDataset(train_s, is_train=True, augmenter=train_augment)
    val_ds   = BrainTumorDataset(val_s,   is_train=False)
    test_ds  = BrainTumorDataset(test_s,  is_train=False)
    _cnt = pd.crosstab([_s["label"] for _s in train_s], [_plane_of(_s["path"]) for _s in train_s])
    _cnt.index = [CLASS_NAMES[i] for i in _cnt.index]
    print("  train images per class x plane:\n" + _cnt.to_string())
    _sampler = None
    if PLANE_BALANCE:
        _pl = [_plane_of(_s["path"]) for _s in train_s]; _lb = [_s["label"] for _s in train_s]
        _cc = pd.Series(list(zip(_lb, _pl))).value_counts().to_dict(); _nc = pd.Series(_lb).value_counts().to_dict()
        _np = {c: len({p for (l, p) in _cc if l == c}) for c in _nc}
        _w = [_nc[l] / (_np[l] * _cc[(l, p)]) for l, p in zip(_lb, _pl)]
        _wts = list(_w)
        print("  plane-balanced sampling ON")
    if SOURCE_BALANCE:
        from collections import Counter
        _lb2 = [_s["label"] for _s in train_s]
        _sr = [_NATIVE_SHAPE.get(_s["path"], (512, 512)) == (512, 512) for _s in train_s]      # True = standard 512 x 512 file
        _n = len(train_s); _cc2 = Counter(zip(_lb2, _sr)); _c1 = Counter(_lb2); _s1 = Counter(_sr)
        _ws = [min(_c1[l] * _s1[r] / (_n * _cc2[(l, r)]), SOURCE_W_CLIP) for l, r in zip(_lb2, _sr)]   # P(class) P(source) / P(class, source)
        _wts = [a * b for a, b in zip((_wts if PLANE_BALANCE else [1.0] * _n), _ws)]
        _eff = pd.DataFrame({"class": [CLASS_NAMES[l] for l in _lb2], "other_size": [not r for r in _sr], "w": _wts})
        _tab = _eff.groupby(["class", "other_size"]).agg(n=("w", "size"), share_before=("w", lambda v: len(v) / _n), share_after=("w", lambda v: v.sum() / _eff.w.sum()))
        print("  source-balanced sampling ON (other_size = file is not 512 x 512):\n" + _tab.to_string(float_format=lambda v: f"{v:.3f}"))
    if PLANE_BALANCE or SOURCE_BALANCE:
        _sampler = WeightedRandomSampler(torch.tensor(_wts, dtype=torch.double), num_samples=len(train_s), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=(_sampler is None), sampler=_sampler,
                               num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True)

    if len(train_s) > TRAIN_EVAL_N:
        train_eval_s, _ = train_test_split(
            train_s, train_size=TRAIN_EVAL_N, random_state=SEED,
            stratify=[s["label"] for s in train_s])
    else:
        train_eval_s = train_s
    train_eval_ds = BrainTumorDataset(train_eval_s, is_train=False)
    train_eval_loader = DataLoader(train_eval_ds, batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=NUM_WORKERS, pin_memory=True)

    model = model_builder(dropout=dropout, **model_kwargs)
    total_params = inference_param_count(model)
    ema = ModelEMA(model, decay=ema_decay)

    class_weights = compute_class_weights(train_s, power=cw_power).to(DEVICE)
    print(f"  class weights (power {cw_power}): {dict(zip(CLASS_NAMES, np.round(class_weights.cpu().numpy(), 3).tolist()))}")
    criterion = CAWLPlus(class_weights, num_classes=NUM_CLASSES,
                          emb_dim=model.head.fc1.out_features, **loss_kwargs).to(DEVICE)

    # No weight decay on BatchNorm / bias / 1-D parameters (incl. the multi-scale softmax weights).
    _decay, _no_decay = [], []
    for _n, _p in model.named_parameters():
        (_no_decay if (_p.ndim <= 1 or _n.endswith(".bias")) else _decay).append(_p)
    optimizer = torch.optim.AdamW(
        [{"params": _decay, "weight_decay": weight_decay},
         {"params": _no_decay, "weight_decay": 0.0}], lr=PEAK_LR)
    scaler = GradScaler() if USE_AMP else None

    history = {"epoch": [], "train_loss": [], "train_acc": [], "train_loss_aug": [], "train_acc_aug": [],
               "val_loss": [], "val_acc": [], "val_auc": [], "val_ece": [], "lr": []}

    _t0 = evaluate_v2(model, train_eval_loader, criterion)
    _v0 = evaluate_v2(model, val_loader, criterion)
    print(f"  epoch 000 (pre-training baseline)  "
          f"train_acc(clean)={_t0['acc']:.4f}  val_acc={_v0['acc']:.4f}  val_auc={_v0['auc']:.4f}")
    history["epoch"].append(0)
    history["train_loss"].append(_t0["loss"]); history["train_acc"].append(_t0["acc"])
    history["train_loss_aug"].append(_t0["loss"]); history["train_acc_aug"].append(_t0["acc"])
    history["val_loss"].append(_v0["loss"]); history["val_acc"].append(_v0["acc"])
    history["val_auc"].append(_v0["auc"]); history["val_ece"].append(_v0["ece"])
    history["lr"].append(0.0)

    best_val_auc = -1.0
    best_state = None
    epochs_no_improve = 0

    for epoch in range(1, total_epochs + 1):
        for g in optimizer.param_groups:
            g["lr"] = lr_at_epoch_v2(epoch)
        criterion.current_epoch = epoch

        train_loss, train_acc = train_one_epoch_mixup(
            model, train_loader, optimizer, criterion, scaler, ema=ema, mixup_alpha=mixup_alpha, cutmix_alpha=cutmix_alpha)
        train_clean = evaluate_v2(model, train_eval_loader, criterion)
        val_metrics = evaluate_v2(model, val_loader, criterion)
        if epoch_callback is not None:
            epoch_callback(epoch, val_metrics)          # e.g. Optuna pruning; may raise to stop this run

        print(f"  epoch {epoch:03d}  lr={lr_at_epoch_v2(epoch):.2e}  "
              f"train_acc(clean)={train_clean['acc']:.4f} train_acc(mixup)={train_acc:.4f}  "
              f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['acc']:.4f} "
              f"val_auc={val_metrics['auc']:.4f} val_ece={val_metrics['ece']:.4f}")

        history["epoch"].append(epoch)
        history["train_loss"].append(train_clean["loss"]); history["train_acc"].append(train_clean["acc"])
        history["train_loss_aug"].append(train_loss); history["train_acc_aug"].append(train_acc)
        history["val_loss"].append(val_metrics["loss"]); history["val_acc"].append(val_metrics["acc"])
        history["val_auc"].append(val_metrics["auc"]); history["val_ece"].append(val_metrics["ece"])
        history["lr"].append(lr_at_epoch_v2(epoch))

        if epoch >= CKPT_MIN_EPOCH:
            if _sel(val_metrics) > best_val_auc:
                best_val_auc = _sel(val_metrics)
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= EARLY_STOP_PATIENCE:
                    print(f"  [EARLY STOP] no validation improvement for {EARLY_STOP_PATIENCE} epochs (counted from epoch {CKPT_MIN_EPOCH}).")
                    break

    if best_state is not None:
        model.load_state_dict(best_state)
    raw_val = evaluate_v2(model, val_loader, criterion)

    ema_model = model_builder(dropout=dropout, **model_kwargs)
    ema.copy_to(ema_model)
    ema_val = evaluate_v2(ema_model, val_loader, criterion)

    if _sel(ema_val) >= _sel(raw_val):
        print(f"  [FINAL WEIGHTS] EMA (val_auc={ema_val['auc']:.4f}) >= "
              f"best-checkpoint (val_auc={raw_val['auc']:.4f}) -> using EMA weights.")
        model = ema_model
    else:
        print(f"  [FINAL WEIGHTS] best-checkpoint (val_auc={raw_val['auc']:.4f}) > "
              f"EMA (val_auc={ema_val['auc']:.4f}) -> keeping best-checkpoint weights.")
        del ema_model

    # --- the inference mode is chosen on VALIDATION only (rule in choose_headline_mode); the test split is scored afterwards ---
    _yv, _pv1, _pv2 = probs_single_and_tta(model, val_loader)
    headline_mode, _val_stats = choose_headline_mode(_yv, _pv1, _pv2)
    print(f"  [HEADLINE MODE, chosen on validation] {headline_mode}  "
          f"(val acc single={_val_stats['val_acc_single']:.4f} tta={_val_stats['val_acc_tta']:.4f}; "
          f"val NLL single={_val_stats['val_nll_single']:.4f} tta={_val_stats['val_nll_tta']:.4f})")
    _yt, _pt1, _pt2 = probs_single_and_tta(model, test_loader)
    np.savez_compressed(OUT_DIR / f"{dataset_name}_predictions.npz", y_val=_yv, val_single=_pv1, val_tta=_pv2,
                        y_test=_yt, test_single=_pt1, test_tta=_pt2,
                        val_paths=np.array([s["path"] for s in val_s]), test_paths=np.array([s["path"] for s in test_s]))
    test_metrics_single = evaluate_v2(model, test_loader, criterion)
    val_metrics_single = evaluate_v2(model, val_loader, criterion)
    val_metrics_tta = evaluate_v2_tta(model, val_loader, criterion)
    print(f"[{dataset_name} [TEST, single-pass]] Accuracy={test_metrics_single['acc']:.4f}  "
          f"F1(macro)={test_metrics_single['f1_mac']:.4f}  AUC={test_metrics_single['auc']:.4f}  ECE={test_metrics_single['ece']:.4f}")
    test_metrics = evaluate_v2_tta(model, test_loader, criterion)
    print(f"[{dataset_name} [TEST, flip-TTA]] Accuracy={test_metrics['acc']:.4f}  "
          f"BalancedAcc={test_metrics['balanced_acc']:.4f}  F1(macro)={test_metrics['f1_mac']:.4f}  "
          f"AUC={test_metrics['auc']:.4f}  MCC={test_metrics['mcc']:.4f}  "
          f"CohenKappa={test_metrics['kappa']:.4f}  ECE={test_metrics['ece']:.4f}")
    print(test_metrics["report"])
    plot_confusion_matrix(test_metrics["cm"], title=f"{dataset_name} Confusion Matrix (flip-TTA)",
                           save_path=str(OUT_DIR / f"{dataset_name}_confusion_matrix.png"))

    torch.save(model.state_dict(), OUT_DIR / f"{dataset_name}.pth")

    return {
        "name": dataset_name, "model": model, "criterion": criterion,
        "total_params": total_params,
        "n_train": len(train_s), "n_val": len(val_s), "n_test": len(test_s),
        "train_samples": train_s, "val_samples": val_s, "test_samples": test_s,
        "test_metrics": test_metrics,
        "test_metrics_single": test_metrics_single, "val_metrics_single": val_metrics_single,
        "val_metrics_tta": val_metrics_tta, "headline_mode": headline_mode, "val_mode_stats": _val_stats,
        "history": history,
        "config": {"dropout": dropout, "weight_decay": weight_decay, "mixup_alpha": mixup_alpha, "cutmix_alpha": cutmix_alpha, "peak_lr": PEAK_LR, "warmup_epochs": WARMUP_EPOCHS_V2,
                   "ema_decay": ema_decay, "epochs": total_epochs, **SMGF_CONFIG, **model_kwargs, "loss_kwargs": {**loss_kwargs, "cw_power": cw_power}},
    }

print("run_experiment() ready (Mixup + EMA + flip-TTA, model-builder-generic).")

In [ ]:
# ===== PATCH: intensity augmentation on the [0,1] image (insert BEFORE the A/B cell, after §13) =====
# Intensity augmentation now acts on the robust-normalised [0,1] image (same scale as the section 17 robustness test).
# Geometry, mixup, dropout, loss, split and schedule are unchanged. Set INT_AUG = False for geometry-only training.
import torch.nn.functional as F

INT_AUG = True

def intensity_aug01(x: torch.Tensor) -> torch.Tensor:
    """x: [H,W] in [0,1] (after robust_normalize). Returns float32 [H,W] in [0,1].
    Uses torch RNG only (DataLoader reseeds it per worker)."""
    x = x.float()                                           # force float32 (the error came from a float64 input)
    H, W = x.shape
    r = lambda lo, hi: lo + (hi - lo) * torch.rand(1).item()
    if torch.rand(1).item() < 0.30:                         # smooth multiplicative bias field
        g = torch.rand(1, 1, 4, 4, dtype=torch.float32)
        g = F.interpolate(g, size=(H, W), mode="bilinear", align_corners=False)[0, 0]
        x = x * (1.0 + 0.25 * (2.0 * g - 1.0))
    if torch.rand(1).item() < 0.50:                         # gamma
        x = x.clamp(0, 1) ** r(0.7, 1.5)
    if torch.rand(1).item() < 0.50:                         # contrast about 0.5
        x = (x - 0.5) * r(0.6, 1.4) + 0.5
    if torch.rand(1).item() < 0.50:                         # brightness shift
        x = x + r(-0.15, 0.15)
    if torch.rand(1).item() < 0.20:                         # light blur
        s = r(0.3, 1.2); k = 5
        ax = torch.arange(k, dtype=torch.float32) - k // 2
        w = torch.exp(-(ax ** 2) / (2.0 * s * s)).float()
        w = w / w.sum()
        ker = (w[:, None] * w[None, :]).float()[None, None]
        xp = F.pad(x.float()[None, None], (k // 2,) * 4, mode="reflect")
        x = F.conv2d(xp, ker.to(xp.dtype))[0, 0]
    if torch.rand(1).item() < 0.40:                         # Gaussian noise, sigma up to 0.05
        x = x + torch.randn(x.shape, dtype=torch.float32) * r(0.01, 0.05)
    return x.clamp(0, 1).float()

if not hasattr(BrainTumorDataset, "_orig_getitem"):
    BrainTumorDataset._orig_getitem = BrainTumorDataset.__getitem__

def _getitem_int01(self, idx):
    if not (self.is_train and INT_AUG):
        return BrainTumorDataset._orig_getitem(self, idx)
    s = self.samples[idx]
    x = torch.from_numpy(np.ascontiguousarray(robust_normalize(_load_resized_gray(s["path"])), dtype=np.float32))
    img = to_3ch_tensor(intensity_aug01(x).numpy())
    mask, valid = get_mask(s)
    if self.augmenter is not None:
        img, mask = self.augmenter(img, mask)               # geometry only (old intensity terms switched off below); mask gets the same geometry
    return {"image": img, "label": torch.tensor(s["label"], dtype=torch.long),
            "class_name": s["class_name"], "path": s["path"],
            "mask": mask, "mask_valid": torch.tensor(valid, dtype=torch.float32)}

BrainTumorDataset.__getitem__ = _getitem_int01

# old tensor-space intensity terms off, so intensity is not augmented twice
train_augment.p_jitter = 0.0; train_augment.p_gamma = 0.0; train_augment.p_noise = 0.0; train_augment.p_blur = 0.0
print("Intensity augmentation on [0,1] image: ON (geometry unchanged, old tensor-space intensity terms off).")

### Recipe (profile "final")
The settings that `run_experiment()` reads, the argument set `FINAL_RUN_KW`, and the `build_smgf_mscaeca` name used by the baseline and checkpoint notebooks.

In [ ]:
# ===== The training recipe: profile "final" of the SMGF_Net_7 notebook (identical in every notebook) =====
# 60 epochs, early-stopping patience 8 counted from epoch 30, checkpoint and early stopping on validation loss, 10 % grouped validation split,
# Mixup 0.2 + CutMix 1.0 (alternating per batch), no focal factor, dropout 0.4, source-balanced sampling, lesion-guided head (SMGF-Net only).
FINAL_PROFILE = dict(dropout=0.4, mixup=0.2, cutmix=CUTMIX_ALPHA, focal=0.0, erase=0.0, balance=False, source_balance=True,
                     select="val_loss", patience=8, val_frac=0.10, min_epoch=30)


def apply_final_profile():
    """Sets the globals that run_experiment() reads. Call it again before any run that follows a cell that changed them."""
    global EARLY_STOP_PATIENCE, SELECT_BY, PLANE_BALANCE, SOURCE_BALANCE, VAL_FRAC, CKPT_MIN_EPOCH
    c = FINAL_PROFILE
    EARLY_STOP_PATIENCE = c["patience"]; SELECT_BY = c["select"]; PLANE_BALANCE = c["balance"]
    SOURCE_BALANCE = c["source_balance"]; VAL_FRAC = c["val_frac"]; CKPT_MIN_EPOCH = c["min_epoch"]
    train_augment.p_erase = c["erase"]


apply_final_profile()
FINAL_RUN_KW = dict(dropout=FINAL_PROFILE["dropout"], mixup_alpha=FINAL_PROFILE["mixup"], cutmix_alpha=FINAL_PROFILE["cutmix"],
                    loss_kwargs={"focal_gamma": FINAL_PROFILE["focal"]})


def build_smgf_mscaeca(**kw):
    """Name used by the baseline / ablation / checkpoint notebooks. attention="none" replaces MS-CA-ECA by the identity (ablation arm)."""
    att = kw.pop("attention", "msca_eca")
    m = build_smgfnet(**kw)
    if att == "none":
        m.attn = nn.Identity()
    return m


print("final profile applied:", FINAL_PROFILE)

In [ ]:
# ===== pHash helpers (review: pHash finding) =========================================================
# The training split in this notebook is the SMGF_Net_7 one (near-duplicate clean at thumbnail correlation 0.98 + grouped validation split).
# These helpers add the perceptual-hash audit: matched vs unmatched accuracy, pHash groups. Set USE_GROUP_SPLIT = True to ALSO drop
# training images that share a pHash cluster with a test image (an extra, stricter leakage filter used by split_train_val_group_aware / drop_train_near_test).
# BRISC2025 has no patient ID, so near-duplicate images are clustered with a 64-bit DCT perceptual hash
# (same hash as notebook 08). All images of one cluster stay on the same side of every split, and
# training images that share a cluster with a TEST image are removed (TEST_LEAK_POLICY).
# PHASH_GROUP_THR: calibrate it with notebook 08 (cell "calibration") and copy the value here.
from sklearn.model_selection import StratifiedGroupKFold

USE_GROUP_SPLIT  = False
PHASH_GROUP_THR  = 6                  # Hamming distance (of 64 bits) at or below which two images count as one group
TEST_LEAK_POLICY = "drop_train"       # "drop_train": remove train images grouped with a test image | "report_only"
GROUP_VAL_FOLDS  = 7                  # one fold of 7 (14.3 %) of the training pool becomes validation
MAX_GROUP_FRACTION_WARN = 0.05        # warn if one cluster holds more than 5 % of the pool (threshold too loose)

_PHASH_CACHE = {}
_GROUP_CACHE = {}
_POPCOUNT8 = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


def phash64_of(path):
    """64-bit DCT perceptual hash as a Python int (None if the file cannot be read). Cached per path."""
    key = str(path)
    if key not in _PHASH_CACHE:
        img = cv2.imread(key, cv2.IMREAD_GRAYSCALE)
        if img is None:
            _PHASH_CACHE[key] = None
        else:
            img = cv2.resize(img, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)
            low = cv2.dct(img)[:8, :8].flatten()
            med = np.median(low[1:])                       # ignore the DC term
            _PHASH_CACHE[key] = int(sum(1 << i for i, b in enumerate(low > med) if b))
    return _PHASH_CACHE[key]


def hash_array(paths):
    h = [phash64_of(p) for p in paths]
    ok = np.array([x is not None for x in h], dtype=bool)
    return np.array([x if x is not None else 0 for x in h], dtype=np.uint64), ok


def hamming_matrix_chunks(ha, hb, chunk=256):
    """Yield (row_offset, distance block) for every chunk of rows of ha against all of hb."""
    for s in range(0, len(ha), chunk):
        x = (ha[s:s + chunk, None] ^ hb[None, :]).view(np.uint8).reshape(-1, len(hb), 8)
        yield s, _POPCOUNT8[x].sum(-1).astype(np.int16)


def nearest_hamming(ha, hb, exclude_self=False):
    """For each hash in ha: (smallest Hamming distance to any hash in hb, index of that neighbour)."""
    dist = np.empty(len(ha), dtype=np.int16); idx = np.empty(len(ha), dtype=np.int64)
    for s, d in hamming_matrix_chunks(ha, hb):
        if exclude_self:
            d[np.arange(d.shape[0]), s + np.arange(d.shape[0])] = 99
        j = d.argmin(1)
        dist[s:s + len(d)] = d[np.arange(len(d)), j]; idx[s:s + len(d)] = j
    return dist, idx


def build_phash_groups(paths, thr):
    """Single-linkage clusters: two images share a group if their pHash distance is <= thr. Returns one id per path."""
    h, ok = hash_array(paths)
    n = len(paths); parent = np.arange(n)

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a

    ok_idx = np.nonzero(ok)[0]; hk = h[ok_idx]
    for s, d in hamming_matrix_chunks(hk, hk):
        ii, jj = np.nonzero(d <= thr)
        ii = ii + s
        keep = ii < jj
        for a, b in zip(ok_idx[ii[keep]], ok_idx[jj[keep]]):
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[rb] = ra
    return np.array([find(i) for i in range(n)])


def _group_ids(pool_paths):
    key = (len(pool_paths), pool_paths[0], pool_paths[-1], PHASH_GROUP_THR)
    if key not in _GROUP_CACHE:
        _GROUP_CACHE[key] = build_phash_groups(pool_paths, PHASH_GROUP_THR)
    return _GROUP_CACHE[key]


def drop_train_near_test(train_pool, test_s, verbose=True):
    """Remove every training image that sits in a pHash group with a test image."""
    if not (USE_GROUP_SPLIT and TEST_LEAK_POLICY == "drop_train"):
        return list(train_pool)
    paths = [s["path"] for s in train_pool] + [s["path"] for s in test_s]
    gid = _group_ids(paths); n_tr = len(train_pool)
    test_groups = set(gid[n_tr:].tolist())
    keep = [g not in test_groups for g in gid[:n_tr]]
    kept = [s for s, k in zip(train_pool, keep) if k]
    if verbose:
        print(f"[group split] pHash thr={PHASH_GROUP_THR}: removed {n_tr - len(kept)} of {n_tr} training images "
              f"that share a cluster with a test image ({100 * (n_tr - len(kept)) / n_tr:.2f} %)")
    return kept


def split_train_val_group_aware(all_train, test_s, seed, val_frac=0.15, verbose=True):
    """Drop-in for  train_test_split(all_train, test_size=0.15, random_state=seed, stratify=labels).
    Validation images never share a pHash cluster with a training image, and (policy "drop_train") no
    training or validation image shares a cluster with a test image."""
    if not USE_GROUP_SPLIT:
        return train_test_split(all_train, test_size=val_frac, random_state=seed,
                                stratify=[s["label"] for s in all_train])
    paths = [s["path"] for s in all_train] + [s["path"] for s in test_s]
    gid = _group_ids(paths); n_tr = len(all_train)
    g_tr, g_te = gid[:n_tr], gid[n_tr:]
    test_groups = set(g_te.tolist())
    leaky = np.array([g in test_groups for g in g_tr])
    keep = ~leaky if TEST_LEAK_POLICY == "drop_train" else np.ones(n_tr, dtype=bool)
    kept = [s for s, k in zip(all_train, keep) if k]; kept_g = g_tr[keep]
    y = np.array([s["label"] for s in kept])
    sgkf = StratifiedGroupKFold(n_splits=GROUP_VAL_FOLDS, shuffle=True, random_state=seed)
    tr_idx, va_idx = next(iter(sgkf.split(np.zeros(len(kept)), y, groups=kept_g)))
    assert not (set(kept_g[tr_idx].tolist()) & set(kept_g[va_idx].tolist())), "a pHash group spans train and validation"
    train_s = [kept[i] for i in tr_idx]; val_s = [kept[i] for i in va_idx]
    if verbose:
        sizes = np.bincount(np.unique(gid, return_inverse=True)[1])
        print(f"[group split] seed={seed} thr={PHASH_GROUP_THR} policy={TEST_LEAK_POLICY}: "
              f"{len(np.unique(gid))} clusters in {len(gid)} images, largest cluster {sizes.max()} images; "
              f"train/test clusters shared: {len(set(g_tr.tolist()) & test_groups)}; "
              f"dropped from train {int(leaky.sum()) if TEST_LEAK_POLICY == 'drop_train' else 0}; "
              f"-> train {len(train_s)}  val {len(val_s)}")
        if sizes.max() > MAX_GROUP_FRACTION_WARN * len(gid):
            print(f"[group split] WARNING: one cluster holds {sizes.max()} images "
                  f"(> {MAX_GROUP_FRACTION_WARN:.0%} of the pool) -- the threshold is too loose; lower PHASH_GROUP_THR")
    pd.DataFrame({"path": paths, "group": gid,
                  "split": ["train_pool"] * n_tr + ["test"] * len(g_te)}).to_csv(OUT_DIR / "group_assignment.csv", index=False)
    return train_s, val_s


# ---- matched vs unmatched accuracy (used by 01 and 07) ----
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n; den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return float(c - h), float(c + h)


def matched_unmatched_accuracy(test_paths, y_true, y_pred, ref_paths, thr_list=(0, 2, 4, 6, 8, 10), ref_name="reference"):
    """Accuracy of test images that have / do not have a pHash neighbour within thr in ref_paths."""
    ht, okt = hash_array(test_paths); hr, okr = hash_array(ref_paths)
    d, _ = nearest_hamming(ht, hr)
    d = np.where(okt, d, 99)
    correct = (np.asarray(y_true) == np.asarray(y_pred))
    rows = []
    for t in thr_list:
        for name, m in (("matched", d <= t), ("unmatched", d > t), ("all", np.ones(len(d), bool))):
            n = int(m.sum()); k = int(correct[m].sum())
            lo, hi = wilson_ci(k, n)
            rows.append({"reference": ref_name, "threshold": t, "subset": name, "n": n, "errors": n - k,
                         "accuracy": (k / n) if n else float("nan"), "wilson_lo": lo, "wilson_hi": hi})
    return pd.DataFrame(rows), d


print(f"Group-aware split ready: USE_GROUP_SPLIT={USE_GROUP_SPLIT}, PHASH_GROUP_THR={PHASH_GROUP_THR}, "
      f"TEST_LEAK_POLICY={TEST_LEAK_POLICY}")

## §2 · Arms and runner

In [ ]:

# ---- helpers: paired test-set predictions with flip-TTA (not returned by run_experiment) ----
@torch.no_grad()
def predict_tta(model, samples):
    ds = BrainTumorDataset(samples, is_train=False)
    dl = DataLoader(ds, batch_size=64, shuffle=False, num_workers=NUM_WORKERS)
    model.eval(); P1=[]; P2=[]; Y=[]
    for b in dl:
        x=b["image"].to(DEVICE); y=b["label"]
        p1=torch.softmax(model(x)["logits"],1); p2=torch.softmax(model(torch.flip(x,dims=[3]))["logits"],1)
        P1.append(p1.cpu().numpy()); P2.append(p2.cpu().numpy()); Y.append(y.numpy())
    P1=np.concatenate(P1); P2=np.concatenate(P2)
    return np.concatenate(Y), P1, (P1+P2)/2

def mcnemar_exact(correct_a, correct_b):
    """Exact two-sided McNemar test on paired correctness vectors. Returns (b, c, p): b = A right/B wrong, c = A wrong/B right."""
    from scipy.stats import binomtest
    b=int(np.sum(correct_a & ~correct_b)); c=int(np.sum(~correct_a & correct_b))
    n=b+c
    p=1.0 if n==0 else binomtest(min(b,c), n, 0.5, alternative="two-sided").pvalue
    return b,c,float(p)

def paired_bootstrap_diff(correct_a, correct_b, n_boot=2000, seed=0):
    rng=np.random.default_rng(seed); n=len(correct_a); d=[]
    for _ in range(n_boot):
        idx=rng.integers(0,n,n); d.append(correct_a[idx].mean()-correct_b[idx].mean())
    return float(np.mean(d)), float(np.percentile(d,2.5)), float(np.percentile(d,97.5))

def count_params(m, exclude=()):
    return sum(p.numel() for n,p in m.named_parameters() if not any(n.startswith(e) for e in exclude))

def efficiency(model, n_iter=200, warm=30):
    from thop import profile as thop_profile
    import copy
    x=torch.randn(1,3,IMG_SIZE,IMG_SIZE,device=DEVICE)
    class _W(nn.Module):
        def __init__(s,m): super().__init__(); s.m=m
        def forward(s,z): return s.m(z)["logits"]
    # thop restores the wrapper's train() flag on exit, which would flip the real model back to train mode -> profile a copy
    _copy = copy.deepcopy(model).eval()
    flops,_=thop_profile(_W(_copy).eval(), inputs=(x,), verbose=False)
    del _copy
    model.eval()
    with torch.no_grad():
        for _ in range(warm): model(x)
        torch.cuda.synchronize() if DEVICE=="cuda" else None
        ts=[]
        for _ in range(n_iter):
            t=time.perf_counter(); model(x)
            torch.cuda.synchronize() if DEVICE=="cuda" else None
            ts.append((time.perf_counter()-t)*1000)
    size_mb=sum(p.numel()*4 for p in model.parameters())/1e6
    return {"flops_G":flops/1e9, "latency_ms_mean":float(np.mean(ts)), "latency_ms_sd":float(np.std(ts)), "size_MB":size_mb}

ARMS = {
    "Full (SMGF-Net + MS-CA-ECA)": dict(),
    "Plain backbone (spatial branch only; no morphology, fusion, or attention)": dict(use_spatial=True, use_morphology=False, use_gated_fusion=False, attention="none"),
}
if RUN_REMOVAL_ARMS:
    ARMS.update({
        "- Spatial branch":  dict(use_spatial=False),
        "- Morphology branch": dict(use_morphology=False),
        "- Gated Fusion (50/50 average)": dict(use_gated_fusion=False),
        "- MS-CA-ECA": dict(attention="none"),
        "- Lesion head (no mask supervision, no mask-aware CutMix)": dict(use_lesion_head=False),   # what the baselines in 02 and 03 effectively get
    })
ARM_LOSS = {}      # arm name -> loss_kwargs overrides (CAWL+ breakdown); model arms leave it empty
if RUN_MSCAECA_PARTS:
    # Breakdown of MS-CA-ECA = multi-scale context (dilated depthwise 3x3, d = 1/2/3) + coordinate attention + ECA, in that order.
    # A switched-off part is an identity pass-through (see MSCAECA); the residual connection and everything else stay unchanged.
    ARMS.update({
        "- MS-CA-ECA part: multi-scale context": dict(ms_kwargs=dict(use_multiscale=False)),
        "- MS-CA-ECA part: coordinate attention": dict(ms_kwargs=dict(use_coord=False)),
        "- MS-CA-ECA part: ECA": dict(ms_kwargs=dict(use_eca=False)),
    })
if RUN_MSCAECA_ONLY_ARMS:
    ARMS.update({
        "MS-CA-ECA only: multi-scale context": dict(ms_kwargs=dict(use_coord=False, use_eca=False)),
        "MS-CA-ECA only: coordinate attention": dict(ms_kwargs=dict(use_multiscale=False, use_eca=False)),
        "MS-CA-ECA only: ECA": dict(ms_kwargs=dict(use_multiscale=False, use_coord=False)),
    })
if RUN_LOSS_ARMS:
    # Breakdown of CAWL+ = class weights (inverse frequency) + label smoothing 0.05 + confidence penalty (lambda_max 0.3, annealed)
    # + prototype margin (beta 0.1). Same model, same everything else; only the loss changes.
    _LOSS = {
        "Loss: plain cross-entropy": dict(cw_power=0.0, label_smoothing=0.0, lambda_max=0.0, beta=0.0),
        "Loss: - class weights": dict(cw_power=0.0),
        "Loss: - label smoothing": dict(label_smoothing=0.0),
        "Loss: - confidence penalty": dict(lambda_max=0.0),
        "Loss: - prototype margin": dict(beta=0.0),
    }
    for _k, _v in _LOSS.items():
        ARMS[_k] = dict(); ARM_LOSS[_k] = _v
RES_PATH = OUT_DIR/"plain_backbone_results.csv"; PRED_PATH = OUT_DIR/"plain_backbone_predictions.npz"
EXPECT_RESUMED_RUNS = 0   # runs finished in the earlier sessions; 0 = do not check
if RESUME_DIR:
    import shutil
    _norm = lambda p: p.name.replace(" ", "").lower()
    _csvs = [p for p in Path(RESUME_DIR).glob("**/*") if p.is_file() and _norm(p) == "plain_backbone_results.csv"]
    _pairs = []
    for _c in _csvs:
        _z = [p for p in _c.parent.iterdir() if p.is_file() and _norm(p) == "plain_backbone_predictions.npz"]
        if _z: _pairs.append((len(pd.read_csv(_c)), _c, _z[0]))
    if not _pairs:
        raise FileNotFoundError(f"no plain_backbone_results.csv + plain_backbone_predictions.npz pair under RESUME_DIR = {RESUME_DIR}. Stopping now so no GPU time is spent re-running finished work.")
    _n, _c, _z = max(_pairs, key=lambda t: t[0])
    shutil.copy(_c, RES_PATH); shutil.copy(_z, PRED_PATH)
    print(f"[resume] using {_c.parent} ({_n} runs)")
rows=[]; preds={}
for base in [OUT_DIR]:
    if base and (base/"plain_backbone_results.csv").exists() and (base/"plain_backbone_predictions.npz").exists():
        rows=pd.read_csv(base/"plain_backbone_results.csv").to_dict("records"); z=np.load(base/"plain_backbone_predictions.npz", allow_pickle=True); preds={k:z[k] for k in z.files}; break
done={(r["arm"], r["seed"]) for r in rows}
print(f"[resume] {len(rows)} finished runs loaded (RESUME_DIR={RESUME_DIR})")
if RESUME_DIR and len(rows) < EXPECT_RESUMED_RUNS:
    raise RuntimeError(f"only {len(rows)} finished runs loaded, expected {EXPECT_RESUMED_RUNS}: the attached files are the older ones. Stopping before any training.")

for seed in SEED_LIST:
    for arm, kw in ARMS.items():
        if (arm, seed) in done: print("skip (done):", arm, seed); continue
        if not can_start_run(): continue
        _t_run = time.time()
        SEED = seed                  # the seed also re-draws the grouped validation split (test split fixed)
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        apply_final_profile()
        res = run_experiment(f"arm_{abs(hash(arm))%10**6}_s{seed}", BRISC_TRAIN_ROOT, BRISC_TEST_ROOT, BRISC_FOLDER_TO_CLASS,
                             model_builder=build_smgf_mscaeca, model_kwargs=kw,
                             **{**FINAL_RUN_KW, "loss_kwargs": {**FINAL_RUN_KW["loss_kwargs"], **ARM_LOSS.get(arm, {})}})
        m=res["model"]; y,p1,pt=predict_tta(m,res["test_samples"]); tm=res["test_metrics"]
        rows.append({"arm":arm,"seed":seed,"params":res["total_params"],"acc_tta":float((pt.argmax(1)==y).mean()),"acc_single":float((p1.argmax(1)==y).mean()),
                     "f1_macro_tta":tm["f1_mac"],"auc_tta":tm["auc"],"ece_tta":tm["ece"],"n_test":len(y)})
        preds[f"{arm}|{seed}|y"]=y; preds[f"{arm}|{seed}|ptta"]=pt
        pd.DataFrame(rows).to_csv(RES_PATH,index=False); np.savez_compressed(PRED_PATH,**preds)
        print(f"[saved] {arm} seed {seed}: acc_tta={rows[-1]['acc_tta']:.4f}  acc_single={rows[-1]['acc_single']:.4f}  params={res['total_params']:,}")
        _RUN_H.append((time.time() - _t_run) / 3600)
        del m,res; torch.cuda.empty_cache()


## §3 · Summary and paired tests against the freshly trained full model

In [ ]:
df=pd.DataFrame(rows); print(df.round(4).to_string())
full="Full (SMGF-Net + MS-CA-ECA)"; out=[]
for seed in sorted(df["seed"].unique()):
    if f"{full}|{seed}|y" not in preds: continue
    y=preds[f"{full}|{seed}|y"]; ca=preds[f"{full}|{seed}|ptta"].argmax(1)==y
    for arm in df["arm"].unique():
        if arm==full or f"{arm}|{seed}|y" not in preds: continue
        cb=preds[f"{arm}|{seed}|ptta"].argmax(1)==y
        b,cc,p=mcnemar_exact(ca,cb); d,lo,hi=paired_bootstrap_diff(ca,cb)
        out.append({"seed":seed,"arm":arm,"acc_full":ca.mean(),"acc_arm":cb.mean(),"full_right_arm_wrong":b,"full_wrong_arm_right":cc,"mcnemar_p":p,"acc_diff_full_minus_arm":d,"ci_lo":lo,"ci_hi":hi})
mc=pd.DataFrame(out); mc.to_csv(OUT_DIR/"plain_backbone_mcnemar.csv",index=False); print(mc.round(4).to_string())
fs=df[df["arm"]==full]["acc_tta"].values
if len(fs)>1: print(f"\nFull model across seeds: mean {fs.mean():.4f}, SD {fs.std(ddof=1):.4f}")
print("\nFor the paper: report the plain-backbone row next to the full model from THIS run, and quote the full-model accuracy of this run (not 96.28 from the earlier run) when computing deltas.")


# ---- mean +- SD over seeds per arm, and the paired difference to the full model on the same seed ----
agg = df.groupby("arm", sort=False)["acc_tta"].agg(n="count", mean="mean", sd=lambda v: v.std(ddof=1) if len(v) > 1 else float("nan"))
fm = df[df["arm"] == full].set_index("seed")["acc_tta"]
agg["delta_vs_full_mean"] = [float((df[df["arm"] == a].set_index("seed")["acc_tta"] - fm).dropna().mean()) for a in agg.index]
agg["seeds_where_arm_not_worse"] = [int(((df[df["arm"] == a].set_index("seed")["acc_tta"] - fm).dropna() >= 0).sum()) for a in agg.index]
agg.to_csv(OUT_DIR / "ablation_arm_summary.csv"); print("\nPer-arm summary over seeds (flip-TTA accuracy):"); print(agg.round(4).to_string())
print("\nWith 3 seeds a difference is only worth reporting if it is clearly larger than the seed SD of the full model; notebook 09 adds Holm-corrected paired tests.")
